In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vinaymandal/nifty50-dataset-2000-2026")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\vedant\.cache\kagglehub\datasets\vinaymandal\nifty50-dataset-2000-2026\versions\1


In [3]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.data_loader import load_nifty_data
from src.validation import validate_data, identify_extreme_returns

In [4]:
DATA_PATH = ROOT / "data" / "raw" / "nifty50.csv"

df = load_nifty_data(DATA_PATH)
df = df.sort_values("Date").reset_index(drop=True)
df.index

RangeIndex(start=0, stop=4585, step=1)

In [5]:
df.head()

,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805


In [6]:
report = validate_data(df)

report

{'rows': 4585,
 'date_validation': {'missing_dates': 0,
  'duplicate_dates': 0,
  'sorted': True,
  'min_date': Timestamp('2007-09-17 00:00:00'),
  'max_date': Timestamp('2026-05-29 00:00:00')},
 'ohlc_validation': {'missing_values': {'Open': 0,
   'High': 0,
   'Low': 0,
   'Close': 0},
  'non_positive_values': {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0},
  'high_less_than_low': 0,
  'open_outside_range': 0,
  'close_outside_range': 0}}

In [7]:
df.describe()

,Date,Open,High,Low,Close
count,4585,4585.000000,4585.000000,4585.000000,4585.000000
mean,2017-01-28 14:53:31.341330,11211.588126,11270.584393,11137.269332,11205.282231
min,2007-09-17 00:00:00,2553.600098,2585.300049,2252.750000,2524.199951
25%,2012-05-25 00:00:00,5707.299805,5744.950195,5671.250000,5704.200195
50%,2017-02-07 00:00:00,8827.950195,8883.000000,8770.200195,8808.900391
75%,2021-10-05 00:00:00,16270.049805,16338.750000,16172.599609,16258.250000
max,2026-05-29 00:00:00,26333.699219,26373.199219,26210.050781,26328.550781
std,NaN,6572.792483,6593.037510,6546.850415,6570.466637


In [8]:
extreme = identify_extreme_returns(
    df, threshold=0.05
)

extreme

,Date,Open,High,Low,Close,DailyReturn
25,2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195,0.055884
86,2008-01-21,5705.000000,5705.000000,4977.100098,5208.799805,-0.087024
87,2008-01-22,5203.350098,5203.350098,4448.500000,4899.299805,-0.059419
88,2008-01-23,4903.049805,5328.049805,4891.600098,5203.399902,0.062070
90,2008-01-25,5035.049805,5399.250000,5035.049805,5383.350098,0.069515
101,2008-02-11,5120.549805,5126.399902,4803.600098,4857.000000,-0.051432
104,2008-02-14,4944.649902,5220.250000,4944.649902,5202.000000,0.055290
116,2008-03-03,5222.799805,5222.799805,4936.049805,4953.000000,-0.051785
123,2008-03-13,4868.700195,4868.799805,4580.149902,4623.600098,-0.050985
125,2008-03-17,4745.450195,4745.450195,4482.100098,4503.100098,-0.051140


In [9]:
print("Rows:", len(df))
print("Start:", df["Date"].min())
print("End:", df["Date"].max())

Rows: 4585
Start: 2007-09-17 00:00:00
End: 2026-05-29 00:00:00


# 3. Event Detection

In [10]:
import importlib
import src.events

importlib.reload(src.events)

<module 'src.events' from 'c:\\Users\\vedant\\Projects\\algochowk-quant-research\\src\\events.py'>

In [11]:
import numpy as np
import pandas as pd

from src.events import run_event_study

In [12]:
df = (
    df
    .sort_values("Date")
    .reset_index(drop=True)
)

In [13]:
events = run_event_study(
    df,
    threshold=-0.03,
    holding_periods=(1, 3, 5, 10),
    exclude_overlapping=True,
    overlap_window=5,
)

In [14]:
events.shape

(53, 18)

In [15]:
events.columns.tolist()

['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'event_return',
 'entry_date_1',
 'entry_price_1',
 'forward_return_1',
 'entry_date_3',
 'entry_price_3',
 'forward_return_3',
 'entry_date_5',
 'entry_price_5',
 'forward_return_5',
 'entry_date_10',
 'entry_price_10',
 'forward_return_10']

In [16]:
events["period"] = np.where(
    events["Date"] < pd.Timestamp("2020-01-01"),
    "development",
    "oos",
)

In [17]:
events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

In [18]:
event_columns = ['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'event_return',
 'entry_date_1',
 'entry_price_1',
 'forward_return_1',
 'entry_date_3',
 'entry_price_3',
 'forward_return_3',
 'entry_date_5',
 'entry_price_5',
 'forward_return_5',
 'entry_date_10',
 'entry_price_10',
 'forward_return_10',
 'period']

events[event_columns].head()

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
0,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,-0.037469,2007-10-19,5360.350098,-0.027060,2007-10-19,5360.350098,0.021146,2007-10-19,5360.350098,0.038915,2007-10-19,5360.350098,0.094415,development
1,2007-11-21,5778.799805,5790.049805,5530.850098,5561.049805,-0.038030,2007-11-22,5564.649902,-0.008141,2007-11-22,5564.649902,0.030020,2007-11-22,5564.649902,0.009506,2007-11-22,5564.649902,0.067453,development
2,2007-12-17,6037.950195,6039.950195,5740.600098,5777.000000,-0.044761,2007-12-18,5777.600098,-0.006110,2007-12-18,5777.600098,-0.001921,2007-12-18,5777.600098,0.050739,2007-12-18,5777.600098,0.069544,development
3,2008-01-18,5907.750000,5908.750000,5677.000000,5705.299805,-0.035159,2008-01-21,5705.000000,-0.086976,2008-01-21,5705.000000,-0.087923,2008-01-21,5705.000000,-0.056380,2008-01-21,5705.000000,-0.067967,development
4,2008-02-07,5322.549805,5344.600098,5113.850098,5133.250000,-0.035566,2008-02-08,5132.100098,-0.002290,2008-02-08,5132.100098,-0.057257,2008-02-08,5132.100098,0.013620,2008-02-08,5132.100098,0.011633,development


In [19]:
events[event_columns].tail(10)

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
43,2020-05-04,9533.500000,9533.500000,9266.950195,9293.500000,-0.057445,2020-05-05,9429.400391,-0.023734,2020-05-05,9429.400391,-0.024429,2020-05-05,9429.400391,-0.020171,2020-05-05,9429.400391,-0.064283,oos
44,2020-05-18,9158.299805,9158.299805,8806.750000,8823.250000,-0.034323,2020-05-19,8961.700195,-0.009217,2020-05-19,8961.700195,0.016130,2020-05-19,8961.700195,0.007515,2020-05-19,8961.700195,0.113527,oos
45,2020-12-21,13741.900391,13777.500000,13131.450195,13328.400391,-0.031405,2020-12-22,13373.650391,0.006928,2020-12-22,13373.650391,0.028085,2020-12-22,13373.650391,0.041795,2020-12-22,13373.650391,0.061752,oos
46,2021-02-26,14888.599609,14919.450195,14467.750000,14529.150391,-0.037636,2021-03-01,14702.500000,0.004016,2021-03-01,14702.500000,0.036939,2021-03-01,14702.500000,0.016024,2021-03-01,14702.500000,0.015440,oos
47,2021-04-12,14644.650391,14652.500000,14248.700195,14310.799805,-0.035326,2021-04-13,14364.900391,0.009739,2021-04-13,14364.900391,0.017609,2021-04-13,14364.900391,-0.004769,2021-04-13,14364.900391,0.034783,oos
48,2022-02-14,17076.150391,17099.500000,16809.650391,16842.800781,-0.030616,2022-02-15,16933.250000,0.024756,2022-02-15,16933.250000,0.021930,2022-02-15,16933.250000,0.016146,2022-02-15,16933.250000,-0.008229,oos
49,2022-02-24,16548.900391,16705.250000,16203.250000,16247.950195,-0.047781,2022-02-25,16515.650391,0.008643,2022-02-25,16515.650391,0.005467,2022-02-25,16515.650391,-0.016366,2022-02-25,16515.650391,0.006951,oos
50,2024-06-04,23179.500000,23179.500000,21281.449219,21884.500000,-0.059294,2024-06-05,22128.349609,0.022234,2024-06-05,22128.349609,0.052503,2024-06-05,22128.349609,0.051359,2024-06-05,22128.349609,0.062709,oos
51,2025-04-07,21758.400391,22254.000000,21743.650391,22161.599609,-0.032433,2025-04-08,22446.750000,0.003969,2025-04-08,22446.750000,0.017009,2025-04-08,22446.750000,0.044124,2025-04-08,22446.750000,0.080188,oos
52,2026-03-19,23197.750000,23378.699219,22930.349609,23002.150391,-0.032621,2026-03-20,23110.150391,0.000188,2026-03-20,23110.150391,-0.008557,2026-03-20,23110.150391,-0.012572,2026-03-20,23110.150391,0.000584,oos


In [20]:
events.head()

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
0,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,-0.037469,2007-10-19,5360.350098,-0.027060,2007-10-19,5360.350098,0.021146,2007-10-19,5360.350098,0.038915,2007-10-19,5360.350098,0.094415,development
1,2007-11-21,5778.799805,5790.049805,5530.850098,5561.049805,-0.038030,2007-11-22,5564.649902,-0.008141,2007-11-22,5564.649902,0.030020,2007-11-22,5564.649902,0.009506,2007-11-22,5564.649902,0.067453,development
2,2007-12-17,6037.950195,6039.950195,5740.600098,5777.000000,-0.044761,2007-12-18,5777.600098,-0.006110,2007-12-18,5777.600098,-0.001921,2007-12-18,5777.600098,0.050739,2007-12-18,5777.600098,0.069544,development
3,2008-01-18,5907.750000,5908.750000,5677.000000,5705.299805,-0.035159,2008-01-21,5705.000000,-0.086976,2008-01-21,5705.000000,-0.087923,2008-01-21,5705.000000,-0.056380,2008-01-21,5705.000000,-0.067967,development
4,2008-02-07,5322.549805,5344.600098,5113.850098,5133.250000,-0.035566,2008-02-08,5132.100098,-0.002290,2008-02-08,5132.100098,-0.057257,2008-02-08,5132.100098,0.013620,2008-02-08,5132.100098,0.011633,development


In [21]:
df[
    (df["Date"] >= "2007-10-18") &
    (df["Date"] <= "2007-10-30")
][
    ["Date", "Open", "High", "Low", "Close"]
]

,Date,Open,High,Low,Close
22,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000
23,2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805
24,2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000
25,2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195
26,2007-10-24,5477.600098,5577.899902,5419.399902,5496.149902
27,2007-10-25,5499.049805,5605.950195,5469.299805,5568.950195
28,2007-10-26,5564.250000,5716.899902,5513.350098,5702.299805
29,2007-10-29,5708.899902,5922.500000,5708.899902,5905.899902
30,2007-10-30,5917.549805,5976.000000,5833.899902,5868.750000


In [22]:
event = events.iloc[0]

event[
    [
        "Date",
        "event_return",
        "entry_date_1",
        "entry_price_1",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
    ]
]

Date                 2007-10-18 00:00:00
event_return                   -0.037469
entry_date_1         2007-10-19 00:00:00
entry_price_1                5360.350098
forward_return_1                -0.02706
forward_return_3                0.021146
forward_return_5                0.038915
forward_return_10               0.094415
Name: 0, dtype: object

In [23]:
event = events.iloc[0]

event[
    [
        "Date",
        "event_return",
        "entry_date_1",
        "entry_price_1",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
    ]
]

Date                 2007-10-18 00:00:00
event_return                   -0.037469
entry_date_1         2007-10-19 00:00:00
entry_price_1                5360.350098
forward_return_1                -0.02706
forward_return_3                0.021146
forward_return_5                0.038915
forward_return_10               0.094415
Name: 0, dtype: object

In [24]:
df.loc[
    event.name:event.name + 10,
    ["Date", "Open", "High", "Low", "Close"]
]

,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805
5,2007-09-24,4837.149902,4941.149902,4837.149902,4932.200195
6,2007-09-25,4939.100098,4953.899902,4878.149902,4938.850098
7,2007-09-26,4937.600098,4980.850098,4930.350098,4940.500000
8,2007-09-27,4942.700195,5016.399902,4942.700195,5000.549805
9,2007-09-28,4996.450195,5055.799805,4996.450195,5021.350098


1-Day

In [25]:
entry = event["entry_price_1"]

actual = (
    df.loc[event.name + 1, "Close"] / entry
) - 1

print(actual)
print(event["forward_return_1"])

-0.1518837179496405
-0.02705985436141034


3-Day

In [26]:
entry = event["entry_price_1"]

actual = (
    df.loc[event.name + 1, "Close"] / entry
) - 1

print(actual)
print(event["forward_return_1"])

-0.1518837179496405
-0.02705985436141034


5-Day

In [27]:
actual = (
    df.loc[event.name + 3, "Close"] /
    event["entry_price_1"]
) - 1

print(actual)
print(event["forward_return_3"])

-0.11432094579730712
0.02114602509000507


10-Day

In [28]:
actual = (
    df.loc[event.name + 10, "Close"] /
    event["entry_price_1"]
) - 1

print(actual)
print(event["forward_return_10"])

-0.05436210266772712
0.09441549309951536


In [29]:
events.shape

(53, 19)

In [30]:
events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

In [31]:
events[
    [
        "Date",
        "event_return",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
        "period",
    ]
].head(10)

,Date,event_return,forward_return_1,forward_return_3,forward_return_5,forward_return_10,period
0,2007-10-18,-0.037469,-0.027060,0.021146,0.038915,0.094415,development
1,2007-11-21,-0.038030,-0.008141,0.030020,0.009506,0.067453,development
2,2007-12-17,-0.044761,-0.006110,-0.001921,0.050739,0.069544,development
3,2008-01-18,-0.035159,-0.086976,-0.087923,-0.056380,-0.067967,development
4,2008-02-07,-0.035566,-0.002290,-0.057257,0.013620,0.011633,development
5,2008-03-03,-0.051785,-0.019018,-0.037702,-0.018685,-0.085821,development
6,2008-03-13,-0.050985,0.026385,-0.019637,-0.003017,0.023941,development
7,2008-03-31,-0.041987,0.000824,0.007591,0.005395,0.030408,development
8,2008-06-20,-0.034789,-0.019478,-0.022638,-0.049297,-0.077026,development
9,2008-07-01,-0.035589,0.050843,0.030986,0.023939,-0.008780,development


In [32]:
from src.baseline import calculate_baseline_returns

In [33]:
baseline = calculate_baseline_returns(
    df, holding_periods=(1,3,5,10),
)
baseline.head()

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260


In [34]:
baseline.shape

(4585, 5)

In [35]:
baseline[
    [
        "Date",
        "baseline_return_1",
        "baseline_return_3",
        "baseline_return_5",
        "baseline_return_10",
    ]
].head(10)

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260
5,2007-09-24,-0.000051,0.012441,0.026290,0.078587
6,2007-09-25,0.000587,0.016962,0.055330,0.102044
7,2007-09-26,0.011704,0.025543,0.053807,0.117780
8,2007-09-27,0.004984,0.042900,0.037907,0.086421
9,2007-09-28,0.009449,0.037270,0.012666,0.129224


In [36]:
baseline[baseline["Date"] == pd.Timestamp("2007-10-18")]

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
22,2007-10-18,-0.02706,0.021146,0.038915,0.094415


Create the clean baseline population

In [37]:
event_dates = set(
    pd.to_datetime(events["Date"])
)

baseline["Date"] = pd.to_datetime(
    baseline["Date"]
)

baseline_non_event = baseline[
    ~baseline["Date"].isin(event_dates)
].copy()

baseline_non_event = baseline_non_event.reset_index(
    drop=True
)

In [38]:
baseline_non_event.shape

(4532, 5)

In [39]:
baseline_non_event.head()

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260


Add the same development/OOS split

In [40]:
baseline_non_event["period"] = np.where(
    baseline_non_event["Date"] < pd.Timestamp("2020-01-01"),
    "development", "oos",
)

In [41]:
baseline_non_event["period"].value_counts()

period
development    2963
oos            1569
Name: count, dtype: int64

First descriptive comparison

In [42]:
horizons = [1, 3, 5, 10]

event_summary = {}

for h in horizons:
    returns = events[
        f"forward_return_{h}"
    ].dropna()

    event_summary[h] = {
        "n": len(returns),
        "mean": returns.mean(),
        "median": returns.median(),
        "win_rate": (returns > 0).mean(),
        "std": returns.std(),
    }

event_summary

{1: {'n': 53,
  'mean': np.float64(-0.0002900340570973539),
  'median': np.float64(0.0005769980693794974),
  'win_rate': np.float64(0.5283018867924528),
  'std': np.float64(0.024593293392500406)},
 3: {'n': 53,
  'mean': np.float64(0.0017272705743365405),
  'median': np.float64(0.006782467466005526),
  'win_rate': np.float64(0.5849056603773585),
  'std': np.float64(0.04411293695580173)},
 5: {'n': 53,
  'mean': np.float64(0.0074232859972477114),
  'median': np.float64(0.013620136204216315),
  'win_rate': np.float64(0.6037735849056604),
  'std': np.float64(0.04858368612414845)},
 10: {'n': 53,
  'mean': np.float64(0.003312384597923094),
  'median': np.float64(0.015439551096752213),
  'win_rate': np.float64(0.5849056603773585),
  'std': np.float64(0.08359437014274604)}}

In [43]:
baseline_summary = {}

for h in horizons:
    returns = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    baseline_summary[h] = {
        "n": len(returns), "mean": returns.mean(),
        "median": returns.median(), "std": returns.std(),
        "win_rate": (returns > 0).mean(),
    }

baseline_summary

{1: {'n': 4531,
  'mean': np.float64(-0.0005020541394949192),
  'median': np.float64(-0.00046557536931701726),
  'std': np.float64(0.011540635771099492),
  'win_rate': np.float64(0.4747296402560141)},
 3: {'n': 4529,
  'mean': np.float64(0.00037346212555367696),
  'median': np.float64(0.001031978970019054),
  'std': np.float64(0.021617777002603308),
  'win_rate': np.float64(0.526164716272908)},
 5: {'n': 4527,
  'mean': np.float64(0.0011887662208082256),
  'median': np.float64(0.002206488531109052),
  'std': np.float64(0.02826865093858146),
  'win_rate': np.float64(0.5420808482438702)},
 10: {'n': 4522,
  'mean': np.float64(0.0033744152431436645),
  'median': np.float64(0.004832403331914148),
  'std': np.float64(0.03922548898693917),
  'win_rate': np.float64(0.5616983635559487)}}

Put them into one table

In [44]:
comparison_rows = []

for h in horizons:
    event_returns = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    comparison_rows.append({
        "horizon": h,

        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (event_returns > 0).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (baseline_returns > 0).mean(),

        "mean_difference": (
            event_returns.mean()
            - baseline_returns.mean()
        ),
    })

comparison = pd.DataFrame(
    comparison_rows
)

comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4531,-0.000502,-0.000466,0.474730,0.000212
1,3,53,0.001727,0.006782,0.584906,4529,0.000373,0.001032,0.526165,0.001354
2,5,53,0.007423,0.013620,0.603774,4527,0.001189,0.002206,0.542081,0.006235
3,10,53,0.003312,0.015440,0.584906,4522,0.003374,0.004832,0.561698,-0.000062


Run it for every horizon

In [45]:
from src.statistics import bootstrap_mean_ci, bootstrap_mean_difference_ci

bootstrap_results = []

for h in [1, 3, 5, 10]:

    values = events[
        f"forward_return_{h}"
    ].dropna()

    result = bootstrap_mean_ci(values)

    bootstrap_results.append({
        "horizon": h,
        **result,
    })

bootstrap_ci = pd.DataFrame(
    bootstrap_results
)

bootstrap_ci

,horizon,mean,ci_lower,ci_upper,n
0,1,-0.000290,-0.006946,0.006144,53
1,3,0.001727,-0.010213,0.013084,53
2,5,0.007423,-0.005639,0.020451,53
3,10,0.003312,-0.020047,0.024868,53


In [46]:
difference_results = []

for h in [1, 3, 5, 10]:

    event_values = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_values = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    result = bootstrap_mean_difference_ci(
        event_values,
        baseline_values,
    )

    difference_results.append({
        "horizon": h,
        **result,
    })

difference_ci = pd.DataFrame(
    difference_results
)

difference_ci

,horizon,difference,ci_lower,ci_upper
0,1,0.000212,-0.006407,0.006709
1,3,0.001354,-0.011171,0.012577
2,5,0.006235,-0.006690,0.019260
3,10,-0.000062,-0.023338,0.021538


In [47]:
oos_events = events[
    events["period"] == "oos"
].copy()

oos_baseline = baseline_non_event[
    baseline_non_event["period"] == "oos"
].copy()

In [48]:
oos_rows = []

for h in [1,3,5,10]:
    event_returns = oos_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = oos_baseline[
        f"baseline_return_{h}"
    ].dropna()

    oos_rows.append({
        "horizon": h,
        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (
            event_returns > 0
        ).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (
            baseline_returns > 0
        ).mean(),
        "mean_difference": (
            event_returns.mean() - baseline_returns.mean()
        ),
    })

oos_comparison = pd.DataFrame(oos_rows)

oos_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,15,0.005853,0.008019,0.800000,1568,-0.000590,-0.000243,0.483418,0.006443
1,3,15,0.001316,0.014141,0.600000,1566,0.000416,0.001198,0.533206,0.000900
2,5,15,0.003491,0.016024,0.600000,1564,0.001412,0.002273,0.539642,0.002079
3,10,15,0.000207,0.023626,0.733333,1559,0.003926,0.005032,0.569596,-0.003719


In [49]:
development_events = events[
    events["period"] == "development"
].copy()

development_baseline = baseline_non_event[
    baseline_non_event["period"] == "development"
].copy()

In [50]:
development_rows = []

for h in [1,3,5,10]:
    event_returns = development_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = development_baseline[
        f"baseline_return_{h}"
    ].dropna()

    development_rows.append({
        "horizon": h,
        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (
            event_returns > 0
        ).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (
            baseline_returns > 0
        ).mean(),
        "mean_difference": (
            event_returns.mean() - baseline_returns.mean()
        ),
    })

development_comparison = pd.DataFrame(development_rows)

development_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,38,-0.002715,-0.002076,0.421053,2963,-0.000456,-0.000576,0.470132,-0.002259
1,3,38,0.001889,0.005385,0.578947,2963,0.000351,0.000957,0.522443,0.001539
2,5,38,0.008975,0.013162,0.605263,2963,0.001071,0.002182,0.543368,0.007905
3,10,38,0.004538,0.007283,0.526316,2963,0.003084,0.004810,0.557543,0.001454


Now test OOS uncertainty

In [51]:
oos_difference_results = []

for h in [1, 3, 5, 10]:

    event_values = oos_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_values = oos_baseline[
        f"baseline_return_{h}"
    ].dropna()

    result = bootstrap_mean_difference_ci(
        event_values,
        baseline_values,
        n_bootstrap=10_000,
        random_state=42,
    )

    oos_difference_results.append({
        "horizon": h,
        **result,
    })

oos_difference_ci = pd.DataFrame(
    oos_difference_results
)

oos_difference_ci

,horizon,difference,ci_lower,ci_upper
0,1,0.006443,-0.001175,0.013376
1,3,0.000900,-0.015203,0.016305
2,5,0.002079,-0.022899,0.021892
3,10,-0.003719,-0.056803,0.040173


In [52]:
from src.baseline import calculate_baseline_returns, create_strict_baseline

Create the Strict baseline

In [53]:
baseline_strict = create_strict_baseline(
    df=df, events=events, 
    baseline=baseline, window=5,
)

In [54]:
baseline_strict.shape

(4267, 5)

Add the development/OOS labels

In [55]:
baseline_strict["period"] = np.where(
    baseline_strict["Date"] < pd.Timestamp("2020-01-01"),
    "development", "oos",
)

In [56]:
baseline_strict["period"].value_counts()

period
development    2773
oos            1494
Name: count, dtype: int64

Verify the exclusion logic

In [57]:
# Pick the first event:
first_event = events.iloc[0]

first_event["Date"]

Timestamp('2007-10-18 00:00:00')

Now inspect the next six trading observations in df:

In [58]:
event_position = first_event.name

df.loc[
    event_position:event_position + 5,
    ["Date", "Open", "Close"]
]

,Date,Open,Close
0,2007-09-17,4518.450195,4494.649902
1,2007-09-18,4494.100098,4546.200195
2,2007-09-19,4550.250000,4732.350098
3,2007-09-20,4734.850098,4747.549805
4,2007-09-21,4752.950195,4837.549805
5,2007-09-24,4837.149902,4932.200195


Now inspect the next six trading observations in df:

In [59]:
excluded_check = baseline_strict[
    baseline_strict["Date"].isin(
        df.loc[
            event_position:event_position + 5,
            "Date"
        ]
    )
]

excluded_check

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10,period
0,2007-09-17,0.011593,0.056396,0.097483,0.127912,development
1,2007-09-18,0.040020,0.063139,0.085402,0.145168,development
2,2007-09-19,0.002682,0.041680,0.043433,0.100066,development
3,2007-09-20,0.017799,0.039113,0.052094,0.091080,development
4,2007-09-21,0.019650,0.021366,0.038080,0.051260,development
5,2007-09-24,-0.000051,0.012441,0.026290,0.078587,development


Verify we're not accidentally deleting unrelated observations

The strict baseline should only remove windows associated with actual selected events.

In [60]:
print("Original baseline:", len(baseline_non_event))
print("Strict baseline:", len(baseline_strict))
print(
    "Removed:", len(baseline_non_event) - len(baseline_strict)
)
# This tells us how many additional observations were removed by the strict specification.

Original baseline: 4532
Strict baseline: 4267
Removed: 265


**Recalculate the descriptive comparison**

Now compare events against the strict baseline.

In [61]:
strict_rows = []

for h in [1,3,5,10]:
    event_returns = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = baseline_strict[
        f"baseline_return_{h}"
    ].dropna()

    strict_rows.append({
        "horizon": h, "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (event_returns > 0).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (baseline_returns > 0).mean(),

        "mean_difference": (event_returns.mean() - baseline_returns.mean()),
    })

strict_comparison = pd.DataFrame(strict_rows)

strict_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4266,-0.000698,-0.000547,0.467886,0.000408
1,3,53,0.001727,0.006782,0.584906,4264,-0.000092,0.000835,0.521811,0.001819
2,5,53,0.007423,0.013620,0.603774,4262,0.000815,0.001987,0.538949,0.006608
3,10,53,0.003312,0.015440,0.584906,4257,0.002702,0.004402,0.557670,0.000611


**Compare the two baseline specifications**

In [62]:
baseline_comparison = comparison[
    [
        "horizon",
        "baseline_mean",
        "mean_difference",
    ]
].merge(
    strict_comparison[
        [
            "horizon",
            "baseline_mean",
            "mean_difference",
        ]
    ],
    on="horizon",
    suffixes=(
        "_original",
        "_strict",
    ),
)

baseline_comparison

,horizon,baseline_mean_original,mean_difference_original,baseline_mean_strict,mean_difference_strict
0,1,-0.000502,0.000212,-0.000698,0.000408
1,3,0.000373,0.001354,-0.000092,0.001819
2,5,0.001189,0.006235,0.000815,0.006608
3,10,0.003374,-0.000062,0.002702,0.000611


In [63]:
baseline_strict.shape

(4267, 6)

In [64]:
baseline_strict["period"].value_counts()

period
development    2773
oos            1494
Name: count, dtype: int64

In [65]:
strict_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4266,-0.000698,-0.000547,0.467886,0.000408
1,3,53,0.001727,0.006782,0.584906,4264,-0.000092,0.000835,0.521811,0.001819
2,5,53,0.007423,0.013620,0.603774,4262,0.000815,0.001987,0.538949,0.006608
3,10,53,0.003312,0.015440,0.584906,4257,0.002702,0.004402,0.557670,0.000611


In [66]:
baseline_comparison

,horizon,baseline_mean_original,mean_difference_original,baseline_mean_strict,mean_difference_strict
0,1,-0.000502,0.000212,-0.000698,0.000408
1,3,0.000373,0.001354,-0.000092,0.001819
2,5,0.001189,0.006235,0.000815,0.006608
3,10,0.003374,-0.000062,0.002702,0.000611


In [67]:
from src.events import detect_events, calculate_forward_returns

thresholds = [
    -0.02,
    -0.025,
    -0.03,
    -0.035,
    -0.04,
    -0.05,
]

threshold_counts = []

for threshold in thresholds:
    threshold_events = detect_events(
        df=df,
        threshold=threshold,
        overlap_window=5,
    )

    threshold_counts.append({
        "threshold": threshold,
        "event_count": len(threshold_events),
    })

threshold_counts = pd.DataFrame(
    threshold_counts
)

threshold_counts

,threshold,event_count
0,-0.020,123
1,-0.025,75
2,-0.030,53
3,-0.035,35
4,-0.040,26
5,-0.050,15


**Create a reusable threshold-analysis function**

In [68]:
def calculate_threshold_analysis(
    df,
    threshold,
    overlap_window=5,
    strict_window=5,
):
    # --------------------------------------------------
    # 1. Detect events
    # --------------------------------------------------

    threshold_events = detect_events(
        df=df,
        threshold=threshold,
        overlap_window=overlap_window,
    )

    # --------------------------------------------------
    # 2. Calculate forward returns
    # --------------------------------------------------

    threshold_events = calculate_forward_returns(
        df=df,
        events=threshold_events,
        holding_periods=(1, 3, 5, 10),
    )

    # --------------------------------------------------
    # 3. Calculate baseline
    # --------------------------------------------------

    threshold_baseline = (
        calculate_baseline_returns(df)
    )

    # --------------------------------------------------
    # 4. Strict baseline
    # --------------------------------------------------

    threshold_baseline_strict = (
        create_strict_baseline(
            df=df,
            events=threshold_events,
            baseline=threshold_baseline,
            window=strict_window,
        )
    )

    # --------------------------------------------------
    # 5. Development / OOS labels
    # --------------------------------------------------

    threshold_events = threshold_events.copy()

    threshold_baseline_strict = (
        threshold_baseline_strict.copy()
    )

    threshold_events["period"] = np.where(
        threshold_events["Date"]
        < pd.Timestamp("2020-01-01"),
        "development",
        "oos",
    )

    threshold_baseline_strict["period"] = np.where(
        threshold_baseline_strict["Date"]
        < pd.Timestamp("2020-01-01"),
        "development",
        "oos",
    )

    # --------------------------------------------------
    # 6. Compare event vs baseline
    # --------------------------------------------------

    rows = []

    for period in [
        "development",
        "oos",
    ]:

        period_events = threshold_events[
            threshold_events["period"] == period
        ]

        period_baseline = threshold_baseline_strict[
            threshold_baseline_strict["period"]
            == period
        ]

        for h in [1, 3, 5, 10]:

            event_returns = period_events[
                f"forward_return_{h}"
            ].dropna()

            baseline_returns = period_baseline[
                f"baseline_return_{h}"
            ].dropna()

            rows.append({
                "threshold": threshold,
                "period": period,
                "horizon": h,

                "event_n": len(
                    event_returns
                ),

                "event_mean": (
                    event_returns.mean()
                    if len(event_returns)
                    else np.nan
                ),

                "event_median": (
                    event_returns.median()
                    if len(event_returns)
                    else np.nan
                ),

                "event_win_rate": (
                    (event_returns > 0).mean()
                    if len(event_returns)
                    else np.nan
                ),

                "baseline_n": len(
                    baseline_returns
                ),

                "baseline_mean": (
                    baseline_returns.mean()
                    if len(baseline_returns)
                    else np.nan
                ),

                "baseline_median": (
                    baseline_returns.median()
                    if len(baseline_returns)
                    else np.nan
                ),

                "baseline_win_rate": (
                    (baseline_returns > 0).mean()
                    if len(baseline_returns)
                    else np.nan
                ),

                "mean_difference": (
                    event_returns.mean()
                    - baseline_returns.mean()
                    if (
                        len(event_returns)
                        and len(baseline_returns)
                    )
                    else np.nan
                ),
            })

    return (
        threshold_events,
        threshold_baseline_strict,
        pd.DataFrame(rows),
    )

**Run all thresholds**

In [69]:
thresholds = [
    -0.02, -0.025, -0.03,
    -0.035, -0.04, -0.05,
]

threshold_results = []
threshold_datasets = {}

for threshold in thresholds:
    (
        threshold_events,
        threshold_baseline_strict,
        threshold_result,
    ) = calculate_threshold_analysis(
        df=df, threshold=threshold,
    )

    threshold_results.append(
        threshold_result
    )

    threshold_datasets[threshold] = {
        "events": threshold_events,
        "baseline": threshold_baseline_strict,
    }

threshold_analysis = pd.concat(
    threshold_results,
    ignore_index=True,
)

**First inspect event counts by period**

In [70]:
threshold_event_counts = (
    threshold_analysis[
        [
            "threshold",
            "period",
            "event_n",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["threshold", "period"]
    )
)

threshold_event_counts

,threshold,period,event_n
40,-0.050,development,11
44,-0.050,oos,4
32,-0.040,development,20
36,-0.040,oos,6
24,-0.035,development,26
28,-0.035,oos,9
16,-0.030,development,38
20,-0.030,oos,15
8,-0.025,development,54
12,-0.025,oos,21


**Generate the main threshold table**

In [71]:
threshold_summary = (
    threshold_analysis[
        [
            "threshold",
            "period",
            "horizon",
            "event_n",
            "event_mean",
            "baseline_mean",
            "mean_difference",
            "event_win_rate",
            "baseline_win_rate",
        ]
    ]
    .sort_values(
        [
            "period",
            "threshold",
            "horizon",
        ]
    )
)

threshold_summary

,threshold,period,horizon,event_n,event_mean,baseline_mean,mean_difference,event_win_rate,baseline_win_rate
40,-0.050,development,1,11,-0.012855,-0.000429,-0.012426,0.181818,0.469847
41,-0.050,development,3,11,-0.016485,0.000527,-0.017012,0.272727,0.525043
42,-0.050,development,5,11,-0.001600,0.001431,-0.003030,0.363636,0.547530
43,-0.050,development,10,11,-0.012877,0.003537,-0.016414,0.454545,0.560136
32,-0.040,development,1,20,-0.009570,-0.000592,-0.008978,0.250000,0.465810
33,-0.040,development,3,20,0.002539,0.000111,0.002428,0.500000,0.521000
34,-0.040,development,5,20,0.017655,0.000987,0.016668,0.500000,0.545297
35,-0.040,development,10,20,0.006126,0.002909,0.003217,0.550000,0.554669
24,-0.035,development,1,26,-0.013405,-0.000591,-0.012814,0.307692,0.464323
25,-0.035,development,3,26,-0.009554,-0.000130,-0.009424,0.461538,0.518453


**Create an easier-to-read 5-day table**

In [72]:
threshold_5d = (
    threshold_analysis[
        threshold_analysis["horizon"] == 5
    ][
        [
            "threshold",
            "period",
            "event_n",
            "event_mean",
            "baseline_mean",
            "mean_difference",
            "event_win_rate",
            "baseline_win_rate",
        ]
    ]
    .sort_values(
        ["period", "threshold"]
    )
)

threshold_5d

,threshold,period,event_n,event_mean,baseline_mean,mean_difference,event_win_rate,baseline_win_rate
42,-0.050,development,11,-0.001600,0.001431,-0.003030,0.363636,0.547530
34,-0.040,development,20,0.017655,0.000987,0.016668,0.500000,0.545297
26,-0.035,development,26,0.010128,0.000586,0.009542,0.538462,0.541301
18,-0.030,development,38,0.008975,0.000446,0.008530,0.605263,0.539488
10,-0.025,development,54,0.002134,0.000274,0.001859,0.574074,0.534554
2,-0.020,development,92,0.000820,0.000636,0.000185,0.500000,0.540220
46,-0.050,oos,4,-0.001588,0.001631,-0.003219,0.500000,0.540193
38,-0.040,oos,6,-0.010031,0.001602,-0.011633,0.500000,0.539857
30,-0.035,oos,9,-0.009319,0.001948,-0.011267,0.444444,0.540984
22,-0.030,oos,15,0.003491,0.001504,0.001987,0.600000,0.537945


**Don't select a winner**

In [73]:
threshold_analysis.sort_values(
    "mean_difference",
    ascending=False
)

,threshold,period,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
45,-0.050,oos,3,4,0.028425,0.018535,0.500000,1557,0.000585,0.001321,0.535003,0.027841
44,-0.050,oos,1,4,0.021386,0.008107,0.500000,1559,-0.000588,-0.000162,0.486851,0.021974
34,-0.040,development,5,20,0.017655,0.001189,0.500000,2881,0.000987,0.002261,0.545297,0.016668
47,-0.050,oos,10,4,0.015494,0.005763,0.500000,1550,0.003833,0.005100,0.572258,0.011661
11,-0.025,development,10,54,0.011869,0.020327,0.574074,2677,0.001561,0.003925,0.549869,0.010308
26,-0.035,development,5,26,0.010128,0.007451,0.538462,2845,0.000586,0.002037,0.541301,0.009542
36,-0.040,oos,1,6,0.008666,0.010326,0.833333,1547,-0.000574,-0.000177,0.486102,0.009239
18,-0.030,development,5,38,0.008975,0.013162,0.605263,2773,0.000446,0.001980,0.539488,0.008530
12,-0.025,oos,1,21,0.006210,0.008019,0.761905,1457,-0.000642,-0.000296,0.478380,0.006852
20,-0.030,oos,1,15,0.005853,0.008019,0.800000,1493,-0.000673,-0.000280,0.480241,0.006526


In [74]:
oos_difference_results = []

for h in [1, 3, 5, 10]:

    event_values = oos_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_values = oos_baseline[
        f"baseline_return_{h}"
    ].dropna()

    result = bootstrap_mean_difference_ci(
        event_values,
        baseline_values,
        n_bootstrap=10_000,
        random_state=42,
    )

    oos_difference_results.append({
        "horizon": h,
        **result,
    })

oos_difference_ci = pd.DataFrame(
    oos_difference_results
)

oos_difference_ci

,horizon,difference,ci_lower,ci_upper
0,1,0.006443,-0.001175,0.013376
1,3,0.000900,-0.015203,0.016305
2,5,0.002079,-0.022899,0.021892
3,10,-0.003719,-0.056803,0.040173


**Get the primary -3% datasets**

In [75]:
threshold_datasets

{-0.02: {'events':           Date          Open          High           Low         Close  \
  0   2007-10-18   5551.100098   5736.799805   5269.649902   5351.000000   
  1   2007-11-20   5911.250000   5923.700195   5755.799805   5780.899902   
  2   2007-12-17   6037.950195   6039.950195   5740.600098   5777.000000   
  3   2008-01-15   6226.350098   6260.450195   6053.299805   6074.250000   
  4   2008-01-24   5208.000000   5357.200195   4995.799805   5033.450195   
  ..         ...           ...           ...           ...           ...   
  118 2024-08-05  24302.849609  24350.050781  23893.699219  24055.599609   
  119 2024-10-03  25452.849609  25639.449219  25230.300781  25250.099609   
  120 2025-04-07  21758.400391  22254.000000  21743.650391  22161.599609   
  121 2026-03-13  23462.500000  23492.400391  23112.000000  23151.099609   
  122 2026-03-23  22824.349609  22851.699219  22471.250000  22512.650391   
  
       event_return entry_date_1  entry_price_1  forward_return_1 en

In [76]:
primary_events = threshold_datasets[-0.03]["events"]

primary_baseline = threshold_datasets[-0.03]["baseline"]

In [77]:
primary_events.shape

(53, 19)

In [78]:
primary_events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

**Run bootstrap for development and OOS**

In [79]:
from src.statistics import bootstrap_mean_difference, block_bootstrap_mean_difference

In [80]:
bootstrap_rows = []

for period in [
    "development",
    "oos",
]:

    period_events = primary_events[
        primary_events["period"] == period
    ]

    period_baseline = primary_baseline[
        primary_baseline["period"] == period
    ]

    for h in [1, 3, 5, 10]:

        event_returns = period_events[
            f"forward_return_{h}"
        ].dropna()

        baseline_returns = period_baseline[
            f"baseline_return_{h}"
        ].dropna()

        result = bootstrap_mean_difference(
            event_returns=event_returns,
            baseline_returns=baseline_returns,
            n_bootstrap=10_000,
            confidence=0.95,
            random_state=42,
        )

        bootstrap_rows.append({
            "threshold": -0.03,
            "period": period,
            "horizon": h,
            **result,
        })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

**Inspect the Result**

In [81]:
bootstrap_results

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper,p_value
0,-0.03,development,1,38.0,2773.0,-0.002003,0.004384,-0.010615,0.006730,0.6782
1,-0.03,development,3,38.0,2773.0,0.002233,0.007825,-0.013912,0.016780,0.7778
2,-0.03,development,5,38.0,2773.0,0.008530,0.007970,-0.007182,0.023974,0.5181
3,-0.03,development,10,38.0,2773.0,0.002521,0.012342,-0.022439,0.026704,0.8364
4,-0.03,oos,1,15.0,1493.0,0.006526,0.003727,-0.001005,0.013515,0.5171
5,-0.03,oos,3,15.0,1491.0,0.000940,0.008060,-0.015545,0.016401,0.9089
6,-0.03,oos,5,15.0,1489.0,0.001987,0.011564,-0.023013,0.021748,0.8727
7,-0.03,oos,10,15.0,1484.0,-0.003773,0.025091,-0.056700,0.040649,0.8814


Format returns as percentages

In [82]:
bootstrap_display = bootstrap_results.copy()

for col in [
    "observed_difference",
    "bootstrap_se",
    "ci_lower", "ci_upper",
]:
    bootstrap_display[col] *= 100

bootstrap_display

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper,p_value
0,-0.03,development,1,38.0,2773.0,-0.200331,0.438353,-1.061487,0.673042,0.6782
1,-0.03,development,3,38.0,2773.0,0.223276,0.782478,-1.391206,1.678003,0.7778
2,-0.03,development,5,38.0,2773.0,0.852988,0.796971,-0.718248,2.397418,0.5181
3,-0.03,development,10,38.0,2773.0,0.252068,1.234167,-2.243858,2.670444,0.8364
4,-0.03,oos,1,15.0,1493.0,0.652648,0.372664,-0.100454,1.351525,0.5171
5,-0.03,oos,3,15.0,1491.0,0.093977,0.805975,-1.554524,1.640057,0.9089
6,-0.03,oos,5,15.0,1489.0,0.198698,1.156429,-2.301310,2.174775,0.8727
7,-0.03,oos,10,15.0,1484.0,-0.377286,2.509090,-5.669973,4.064940,0.8814


**Also calculate the raw event statistics**

In [83]:
primary_summary = (
    threshold_analysis[
        threshold_analysis["threshold"] == -0.03
    ]
    .sort_values(
        ["period", "horizon"]
    )
)

primary_summary

,threshold,period,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
16,-0.03,development,1,38,-0.002715,-0.002076,0.421053,2773,-0.000712,-0.000724,0.461233,-0.002003
17,-0.03,development,3,38,0.001889,0.005385,0.578947,2773,-0.000343,0.000693,0.516769,0.002233
18,-0.03,development,5,38,0.008975,0.013162,0.605263,2773,0.000446,0.001980,0.539488,0.008530
19,-0.03,development,10,38,0.004538,0.007283,0.526316,2773,0.002018,0.004122,0.552110,0.002521
20,-0.03,oos,1,15,0.005853,0.008019,0.800000,1493,-0.000673,-0.000280,0.480241,0.006526
21,-0.03,oos,3,15,0.001316,0.014141,0.600000,1491,0.000377,0.001140,0.531187,0.000940
22,-0.03,oos,5,15,0.003491,0.016024,0.600000,1489,0.001504,0.002036,0.537945,0.001987
23,-0.03,oos,10,15,0.000207,0.023626,0.733333,1484,0.003980,0.004772,0.568059,-0.003773


**First Debug: compare the exact arrays**

In [84]:
threshold = -0.03
period = "oos"
h = 5

period_events = primary_events[
    primary_events["period"] == period
]

period_baseline = primary_baseline[
    primary_baseline["period"] == period
]

event_returns = period_events[
    f"forward_return_{h}"
].dropna()

baseline_returns = period_baseline[
    f"baseline_return_{h}"
].dropna()

print("Event N:", len(event_returns))
print("Baseline N:", len(baseline_returns))

print(
    "Event mean:",
    event_returns.mean()
)

print(
    "Baseline mean:",
    baseline_returns.mean()
)

print(
    "Difference:",
    event_returns.mean()
    - baseline_returns.mean()
)

Event N: 15
Baseline N: 1489
Event mean: 0.003491122444900922
Baseline mean: 0.0015041444590257135
Difference: 0.0019869779858752083


Then Run

In [85]:
check = bootstrap_mean_difference(
    event_returns=event_returns,
    baseline_returns=baseline_returns,
    n_bootstrap=10_000,
    confidence=0.95,
    random_state=42,
)

check

{'event_n': 15.0,
 'baseline_n': 1489.0,
 'observed_difference': 0.0019869779858752083,
 'bootstrap_se': 0.011564290691808822,
 'ci_lower': -0.02301309809247876,
 'ci_upper': 0.02174775392543411,
 'p_value': 0.8727}

In [86]:
check["observed_difference"]

0.0019869779858752083

**make the pipeline internally consistent**

In [87]:
diagnostic = []

for period in ["development", "oos"]:

    period_events = primary_events[
        primary_events["period"] == period
    ]

    period_baseline = primary_baseline[
        primary_baseline["period"] == period
    ]

    for h in [1, 3, 5, 10]:

        event_returns = period_events[
            f"forward_return_{h}"
        ].dropna()

        baseline_returns = period_baseline[
            f"baseline_return_{h}"
        ].dropna()

        observed = (
            event_returns.mean()
            - baseline_returns.mean()
        )

        bootstrap = bootstrap_mean_difference(
            event_returns,
            baseline_returns,
            n_bootstrap=10_000,
            random_state=42,
        )

        diagnostic.append({
            "period": period,
            "horizon": h,
            "event_n": len(event_returns),
            "baseline_n": len(baseline_returns),
            "direct_difference": observed,
            "bootstrap_difference":
                bootstrap["observed_difference"],
            "difference_check":
                np.isclose(
                    observed,
                    bootstrap[
                        "observed_difference"
                    ],
                ),
        })

diagnostic_df = pd.DataFrame(
    diagnostic
)

diagnostic_df

,period,horizon,event_n,baseline_n,direct_difference,bootstrap_difference,difference_check
0,development,1,38,2773,-0.002003,-0.002003,True
1,development,3,38,2773,0.002233,0.002233,True
2,development,5,38,2773,0.008530,0.008530,True
3,development,10,38,2773,0.002521,0.002521,True
4,oos,1,15,1493,0.006526,0.006526,True
5,oos,3,15,1491,0.000940,0.000940,True
6,oos,5,15,1489,0.001987,0.001987,True
7,oos,10,15,1484,-0.003773,-0.003773,True


**Testing Time-Aware Bootstrap**

In [88]:
event_returns = primary_events[
    primary_events["period"] == "oos"
]["forward_return_5"].dropna()

baseline_returns = primary_baseline[
    primary_baseline["period"] == "oos"
]["baseline_return_5"].dropna()

In [89]:
block_result = block_bootstrap_mean_difference(
    event_returns=event_returns,
    baseline_returns=baseline_returns,
    n_bootstrap=10_000,
    block_length=5,
    random_state=42,
)

block_result

{'event_n': 15.0,
 'baseline_n': 1489.0,
 'observed_difference': 0.0019869779858752083,
 'bootstrap_se': 0.011452397196763617,
 'ci_lower': -0.02262459882534538,
 'ci_upper': 0.022083524518713146}

In [90]:
block_result["observed_difference"]

0.0019869779858752083

**Run the block bootstrap across all primary horizons**

In [91]:
block_bootstrap_rows = []

for period in ["development", "oos"]:
    period_events = primary_events[
        primary_events["period"] == period
    ]

    period_baseline = primary_baseline[
        primary_baseline["period"] == period
    ]

    for h in [1,3,5,10]:
        event_returns = period_events[
            f"forward_return_{h}"
        ].dropna()

        baseline_returns = period_baseline[
            f"baseline_return_{h}"
        ].dropna()

        result = block_bootstrap_mean_difference(
            event_returns=event_returns, baseline_returns=baseline_returns,
            n_bootstrap=10_000, block_length=5, random_state=42,
        )

        block_bootstrap_rows.append({
            "threshold": -0.03, "period": period,
            "horizon": h, **result,
        })

block_bootstrap_results = pd.DataFrame(block_bootstrap_rows)

In [92]:
block_bootstrap_display = (block_bootstrap_results.copy())

for col in [
    "observed_difference", "bootstrap_se",
    "ci_lower", "ci_upper",
]:
    block_bootstrap_display[col] *= 100

block_bootstrap_display

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper
0,-0.03,development,1,38.0,2773.0,-0.200331,0.436070,-1.076661,0.643016
1,-0.03,development,3,38.0,2773.0,0.223276,0.776380,-1.390813,1.660891
2,-0.03,development,5,38.0,2773.0,0.852988,0.808371,-0.728049,2.444135
3,-0.03,development,10,38.0,2773.0,0.252068,1.250142,-2.250339,2.655958
4,-0.03,oos,1,15.0,1493.0,0.652648,0.375639,-0.100856,1.362840
5,-0.03,oos,3,15.0,1491.0,0.093977,0.808577,-1.525327,1.612387
6,-0.03,oos,5,15.0,1489.0,0.198698,1.145240,-2.262460,2.208352
7,-0.03,oos,10,15.0,1484.0,-0.377286,2.477907,-5.624936,4.032807


**Next: Multiple-horizon inference**

Create a primary/secondary labeling system

In [93]:
block_bootstrap_results["analysis_type"] = np.where((
    (block_bootstrap_results["threshold"] == -0.03) & (block_bootstrap_results["horizon"] == 5)
), "primary", "secondary",)

In [94]:
block_bootstrap_results[[
    "period", "horizon", "analysis_type", "observed_difference", "ci_lower", "ci_upper",
]]

,period,horizon,analysis_type,observed_difference,ci_lower,ci_upper
0,development,1,secondary,-0.002003,-0.010767,0.006430
1,development,3,secondary,0.002233,-0.013908,0.016609
2,development,5,primary,0.008530,-0.007280,0.024441
3,development,10,secondary,0.002521,-0.022503,0.026560
4,oos,1,secondary,0.006526,-0.001009,0.013628
5,oos,3,secondary,0.000940,-0.015253,0.016124
6,oos,5,primary,0.001987,-0.022625,0.022084
7,oos,10,secondary,-0.003773,-0.056249,0.040328


**Build a clean Inference Table**

In [95]:
primary_inference = (
    block_bootstrap_results[
        [
            "threshold", "period", "horizon", "event_n",
            "baseline_n", "observed_difference", "bootstrap_se",
            "ci_lower", "ci_upper",
        ]
    ].copy()
)

In [96]:
for col in [
    "observed_difference", "bootstrap_se",
    "ci_lower", "ci_upper",
]:
    primary_inference[col] *= 100

In [97]:
primary_inference

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper
0,-0.03,development,1,38.0,2773.0,-0.200331,0.436070,-1.076661,0.643016
1,-0.03,development,3,38.0,2773.0,0.223276,0.776380,-1.390813,1.660891
2,-0.03,development,5,38.0,2773.0,0.852988,0.808371,-0.728049,2.444135
3,-0.03,development,10,38.0,2773.0,0.252068,1.250142,-2.250339,2.655958
4,-0.03,oos,1,15.0,1493.0,0.652648,0.375639,-0.100856,1.362840
5,-0.03,oos,3,15.0,1491.0,0.093977,0.808577,-1.525327,1.612387
6,-0.03,oos,5,15.0,1489.0,0.198698,1.145240,-2.262460,2.208352
7,-0.03,oos,10,15.0,1484.0,-0.377286,2.477907,-5.624936,4.032807


**Regime Robustness**

In [98]:
from src.regimes import add_volatility_regime

In [99]:
regime_df = add_volatility_regime(
    df, volatility_window=20, regime_lookback=252,
)

**Inspect the Result**

In [100]:
regime_df[
    [
        "Date", "Close", "rolling_volatility", "volatility_regime",
    ]
].head(20)

,Date,Close,rolling_volatility,volatility_regime
0,2007-09-17,4494.649902,NaN,NaN
1,2007-09-18,4546.200195,NaN,NaN
2,2007-09-19,4732.350098,NaN,NaN
3,2007-09-20,4747.549805,NaN,NaN
4,2007-09-21,4837.549805,NaN,NaN
5,2007-09-24,4932.200195,NaN,NaN
6,2007-09-25,4938.850098,NaN,NaN
7,2007-09-26,4940.500000,NaN,NaN
8,2007-09-27,5000.549805,NaN,NaN
9,2007-09-28,5021.350098,NaN,NaN


**Check Regime Distribution**

In [101]:
regime_df["volatility_regime"].value_counts(
    dropna=False
)

volatility_regime
low       1712
high      1339
medium    1282
NaN        252
Name: count, dtype: int64

In [102]:
regime_df["volatility_regime"].value_counts(
    normalize=True
)

volatility_regime
low       0.395107
high      0.309024
medium    0.295869
Name: proportion, dtype: float64

In [103]:
primary_events_regime = (
    primary_events.merge(regime_df[
        [
            "Date", "rolling_volatility",
            "volatility_regime",
        ]
    ], on="Date", how="left")
)

In [104]:
primary_events_regime[
    [
        "Date", "event_return",
        "rolling_volatility", "volatility_regime",
    ]
].head()

,Date,event_return,rolling_volatility,volatility_regime
0,2007-10-18,-0.037469,0.333930,NaN
1,2007-11-21,-0.038030,0.302806,NaN
2,2007-12-17,-0.044761,0.302236,NaN
3,2008-01-18,-0.035159,0.248080,NaN
4,2008-02-07,-0.035566,0.595764,NaN


**Check how many events fall into each regime**

In [105]:
primary_events_regime[
    "volatility_regime"
].value_counts(
    dropna=False
)

volatility_regime
high      25
NaN       13
medium     9
low        6
Name: count, dtype: int64

**Merge regime labels into the baseline too**

In [106]:
primary_baseline_regime = (
    primary_baseline
    .merge(
        regime_df[
            [
                "Date",
                "rolling_volatility",
                "volatility_regime",
            ]
        ],
        on="Date",
        how="left",
    )
)

In [107]:
primary_events_regime.groupby(
    ["period", "volatility_regime"], dropna=False
).size()

period       volatility_regime
development  high                 12
             low                   6
             medium                7
             NaN                  13
oos          high                 13
             medium                2
dtype: int64

**Regime-conditioned analysis**

Use only classified observations

In [108]:
events_regime_valid = primary_events_regime[
    primary_events_regime["volatility_regime"].notna()
].copy()

baseline_regime_valid = primary_baseline_regime[
    primary_baseline_regime["volatility_regime"].notna()
].copy()

In [109]:
events_regime_valid.groupby(
    ["period", "volatility_regime"]
).size()

period       volatility_regime
development  high                 12
             low                   6
             medium                7
oos          high                 13
             medium                2
dtype: int64

**Calculate regime-conditioned statistics**

In [110]:
from src.regime_analysis import calculate_regime_analysis

In [111]:
regime_results = calculate_regime_analysis(
    events=events_regime_valid, baseline=baseline_regime_valid,
    horizons=(1,3,5,10), min_events=5,
)

**Display the results**

In [112]:
regime_display = regime_results.copy()

for col in [
    "event_mean",
    "event_median",
    "event_win_rate",
    "baseline_mean",
    "baseline_median",
    "baseline_win_rate",
    "mean_difference",
]:
    regime_display[col] *= 100

regime_display

,period,regime,horizon,event_n,valid_event_n,baseline_n,valid_baseline_n,event_mean,event_median,event_win_rate,baseline_mean,baseline_median,baseline_win_rate,mean_difference,sufficient_events
0,development,low,1,6,6,1077,1077,-0.384164,-0.887201,50.000000,-0.020622,-0.067973,45.589601,-0.363542,True
1,development,low,3,6,6,1077,1077,-0.218976,-0.752711,50.000000,0.063589,0.092762,53.017642,-0.282565,True
2,development,low,5,6,6,1077,1077,0.797291,0.018710,50.000000,0.188131,0.315906,56.731662,0.609160,True
3,development,low,10,6,6,1077,1077,1.845126,2.154683,50.000000,0.561981,0.676870,59.238626,1.283145,True
4,development,medium,1,7,7,773,773,0.072431,0.442199,57.142857,-0.089816,-0.085788,45.666235,0.162247,True
5,development,medium,3,7,7,773,773,1.319880,-0.124646,42.857143,0.032148,0.106088,52.781371,1.287732,True
6,development,medium,5,7,7,773,773,0.915166,1.608546,57.142857,0.085308,0.119906,52.263907,0.829858,True
7,development,medium,10,7,7,773,773,2.875471,6.578383,71.428571,-0.003410,0.065241,50.840880,2.878880,True
8,development,high,1,12,12,748,748,-0.102733,-0.257532,33.333333,-0.084635,-0.051063,47.727273,-0.018098,True
9,development,high,3,12,12,748,748,0.212781,1.339380,83.333333,-0.052390,-0.003330,49.866310,0.265170,True


Before interpreting the regime effect, calculate the event frequency by regime.

In [113]:
event_frequency = (
    events_regime_valid
    .groupby(
        ["period", "volatility_regime"]
    )
    .size()
    .reset_index(name="event_n")
)

event_frequency

,period,volatility_regime,event_n
0,development,high,12
1,development,low,6
2,development,medium,7
3,oos,high,13
4,oos,medium,2


Then compare it against the number of baseline observations

In [114]:
baseline_frequency = (
    baseline_regime_valid
    .groupby(
        ["period", "volatility_regime"]
    )
    .size()
    .reset_index(name="baseline_n")
)

baseline_frequency

,period,volatility_regime,baseline_n
0,development,high,748
1,development,low,1077
2,development,medium,773
3,oos,high,445
4,oos,low,594
5,oos,medium,455


**Bootstrap regime effects**

In [115]:
from src.regime_analysis import calculate_regime_bootstrap

regime_bootstrap_results = (
    calculate_regime_bootstrap(
        events=events_regime_valid,
        baseline=baseline_regime_valid,
        horizons=(1,3,5,10), min_events=5,
        n_bootstrap=10_000,
        block_length=5, random_state=42,
    )
)

In [116]:
regime_bootstrap_display = (regime_bootstrap_results.copy())

for col in [
    "observed_difference", "bootstrap_se",
    "ci_lower", "ci_upper",
]:
    regime_bootstrap_display[col] *= 100

regime_bootstrap_display

,period,regime,horizon,event_n,baseline_n,sufficient_events,observed_difference,bootstrap_se,ci_lower,ci_upper
0,development,low,1,6,1077,True,-0.363542,0.891951,-2.022359,1.473067
1,development,low,3,6,1077,True,-0.282565,1.200959,-2.460733,2.272000
2,development,low,5,6,1077,True,0.609160,1.760734,-2.668978,4.195122
3,development,low,10,6,1077,True,1.283145,2.514593,-3.581594,6.079202
4,development,medium,1,7,773,True,0.162247,0.747265,-1.357184,1.590443
5,development,medium,3,7,773,True,1.287732,1.854916,-2.178178,4.981437
6,development,medium,5,7,773,True,0.829858,2.879351,-4.650038,6.498140
7,development,medium,10,7,773,True,2.878880,3.217288,-3.744914,8.709569
8,development,high,1,12,748,True,-0.018098,0.782354,-1.446496,1.602487
9,development,high,3,12,748,True,0.265170,1.709878,-3.536222,2.916803


Before the next phase, we should verify why those 252 rows are NaN.

In [117]:
print(regime_df.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'rolling_volatility', 'volatility_regime']


In [118]:
regime_df[
    regime_df["volatility_regime"].isna()
][[
    "Date",
    "rolling_volatility",
]].head(20)

,Date,rolling_volatility
0,2007-09-17,NaN
1,2007-09-18,NaN
2,2007-09-19,NaN
3,2007-09-20,NaN
4,2007-09-21,NaN
5,2007-09-24,NaN
6,2007-09-25,NaN
7,2007-09-26,NaN
8,2007-09-27,NaN
9,2007-09-28,NaN


In [119]:
print(
    regime_df[
        regime_df["volatility_regime"].isna()
    ]["Date"].agg(["min", "max"])
)

min   2007-09-17
max   2008-09-19
Name: Date, dtype: datetime64[us]


**Pre-event Trend Regime**

In [120]:
df_trend = df.copy()

df_trend["ma_20"] = (
    df_trend["Close"]
    .rolling(20)
    .mean()
)

df_trend["ma_50"] = (
    df_trend["Close"]
    .rolling(50)
    .mean()
)

In [121]:
df_trend["ma_20_prev"] = (
    df_trend["ma_20"].shift(1)
)

df_trend["ma_50_prev"] = (
    df_trend["ma_50"].shift(1)
)

**Calculate trend strength**

In [122]:
df_trend["trend_strength"] = (
    df_trend["ma_20_prev"] /
    df_trend["ma_50_prev"]
) - 1

Define three regimes

In [123]:
TREND_BAND = 0.01

import numpy as np

df_trend["trend_regime"] = np.select(
    [
        df_trend["trend_strength"] > TREND_BAND,
        df_trend["trend_strength"] < TREND_BAND,
    ], [
        "uptrend", "downtrend"
    ], default="sideways"
)

In [124]:
df_trend.loc[
    df_trend[
        ["ma_20_prev", "ma_50_prev"]
    ].isna().any(axis=1),
    "trend_regime"
] = np.nan

**Inspect the distribution**

In [125]:
df_trend["trend_regime"].value_counts(dropna=False)

trend_regime
downtrend    2391
uptrend      2144
NaN            50
Name: count, dtype: int64

In [126]:
df_trend["trend_regime"].value_counts(normalize=True, dropna=False)

trend_regime
downtrend    0.521483
uptrend      0.467612
NaN          0.010905
Name: proportion, dtype: float64

In [127]:
df_trend[
    [
        "Date", "Close",
        "ma_20_prev",
        "ma_50_prev",
        "trend_strength",
        "trend_regime",
    ]
].tail(20)

,Date,Close,ma_20_prev,ma_50_prev,trend_strength,trend_regime
4565,2026-04-30,23997.550781,23795.984961,24173.825000,-0.015630,downtrend
4566,2026-05-04,24119.300781,23879.292480,24137.632031,-0.010703,downtrend
4567,2026-05-05,24032.800781,23951.287500,24110.596055,-0.006607,downtrend
4568,2026-05-06,24330.949219,24017.272559,24077.597070,-0.002505,downtrend
4569,2026-05-07,24326.650391,24085.407520,24049.708047,0.001484,downtrend
4570,2026-05-08,24176.150391,24145.557520,24019.854062,0.005233,downtrend
4571,2026-05-11,23815.849609,24154.497559,23994.290078,0.006677,downtrend
4572,2026-05-12,23379.550781,24156.535059,23959.182070,0.008237,downtrend
4573,2026-05-13,23412.599609,24122.982617,23912.513086,0.008802,downtrend
4574,2026-05-14,23689.599609,24101.480078,23872.272070,0.009601,downtrend


In [128]:
trend_lookup = df_trend[
    ["Date", "trend_regime"]
].copy()

In [129]:
events = events.merge(
    trend_lookup,
    on="Date",
    how="left",
    validate="one_to_one",
)

In [130]:
events[
    ["Date", "event_return", "trend_regime", "period"]
].head(20)

,Date,event_return,trend_regime,period
0,2007-10-18,-0.037469,NaN,development
1,2007-11-21,-0.038030,NaN,development
2,2007-12-17,-0.044761,uptrend,development
3,2008-01-18,-0.035159,uptrend,development
4,2008-02-07,-0.035566,downtrend,development
5,2008-03-03,-0.051785,downtrend,development
6,2008-03-13,-0.050985,downtrend,development
7,2008-03-31,-0.041987,downtrend,development
8,2008-06-20,-0.034789,downtrend,development
9,2008-07-01,-0.035589,downtrend,development


In [131]:
events["trend_regime"].value_counts(dropna=False)

trend_regime
downtrend    36
uptrend      15
NaN           2
Name: count, dtype: int64

In [132]:
events.groupby(
    ["period", "trend_regime"],
    dropna=False
).size()

period       trend_regime
development  downtrend       23
             uptrend         13
             NaN              2
oos          downtrend       13
             uptrend          2
dtype: int64

In [133]:
print(events.columns.tolist()) 

['Date', 'Open', 'High', 'Low', 'Close', 'event_return', 'entry_date_1', 'entry_price_1', 'forward_return_1', 'entry_date_3', 'entry_price_3', 'forward_return_3', 'entry_date_5', 'entry_price_5', 'forward_return_5', 'entry_date_10', 'entry_price_10', 'forward_return_10', 'period', 'trend_regime']


In [134]:
events[
    [
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
    ]
].describe()

,forward_return_1,forward_return_3,forward_return_5,forward_return_10
count,53.000000,53.000000,53.000000,53.000000
mean,-0.000290,0.001727,0.007423,0.003312
std,0.024593,0.044113,0.048584,0.083594
min,-0.086976,-0.176390,-0.132302,-0.245130
25%,-0.016079,-0.022601,-0.018685,-0.042632
50%,0.000577,0.006782,0.013620,0.015440
75%,0.012009,0.028085,0.039233,0.065784
max,0.066547,0.086085,0.121156,0.135057


**Run the trend-regime event study**

In [135]:
trend_regime_results = []

for period in ["development", "oos"]:
    for regime in ["downtrend", "uptrend"]:

        regime_events = events[
            (events["period"] == period) &
            (events["trend_regime"] == regime)
        ].copy()

        for h in [1, 3, 5, 10]:

            return_col = f"forward_return_{h}"

            valid = regime_events[return_col].dropna()

            if len(valid) == 0:
                continue

            trend_regime_results.append({
                "period": period,
                "regime": regime,
                "horizon": h,
                "event_n": len(valid),
                "event_mean": valid.mean(),
                "event_median": valid.median(),
                "event_win_rate": (valid > 0).mean(),
            })

trend_regime_results = pd.DataFrame(
    trend_regime_results
)

trend_regime_results

,period,regime,horizon,event_n,event_mean,event_median,event_win_rate
0,development,downtrend,1,23,-0.002395,-0.003289,0.347826
1,development,downtrend,3,23,0.000042,0.003988,0.608696
2,development,downtrend,5,23,0.003548,0.005395,0.521739
3,development,downtrend,10,23,-0.009706,-0.004684,0.478261
4,development,uptrend,1,13,-0.000991,0.000874,0.615385
5,development,uptrend,3,13,0.001513,-0.001246,0.461538
6,development,uptrend,5,13,0.016234,0.016085,0.692308
7,development,uptrend,10,13,0.017986,0.024073,0.538462
8,oos,downtrend,1,13,0.005912,0.008643,0.769231
9,oos,downtrend,3,13,-0.003483,0.005467,0.538462


In [136]:
print(baseline.columns.tolist())
print(baseline.shape)

['Date', 'baseline_return_1', 'baseline_return_3', 'baseline_return_5', 'baseline_return_10']
(4585, 5)


**Phase: Trend-regime event vs baseline**

**1. Attach trend regime to baseline**

In [137]:
baseline_trend = baseline.merge(
    df_trend[["Date", "trend_regime"]],
    on="Date", how="left", validate="one_to_one",
)

baseline_trend["period"] = np.where(
    baseline_trend["Date"] <= events.loc[
        events["period"].eq("development"), "Date"
    ].max(), "development", "oos",
)

In [138]:
baseline_trend[
    ["Date", "trend_regime", "period"]
].head()

,Date,trend_regime,period
0,2007-09-17,NaN,development
1,2007-09-18,NaN,development
2,2007-09-19,NaN,development
3,2007-09-20,NaN,development
4,2007-09-21,NaN,development


In [139]:
baseline_trend.groupby(
    ["period", "trend_regime"],
    dropna=False
).size()

period       trend_regime
development  downtrend       1104
             uptrend          897
             NaN               50
oos          downtrend       1287
             uptrend         1247
dtype: int64

**2. Calculate regime-conditioned comparison**

In [140]:
trend_comparison = []

for period in ["development", "oos"]:
    for regime in ["downtrend", "uptrend"]:

        regime_events = events[
            (events["period"] == period) &
            (events["trend_regime"] == regime)
        ].copy()

        regime_baseline = baseline_trend[
            (baseline_trend["period"] == period) &
            (baseline_trend["trend_regime"] == regime)
        ].copy()

        for h in [1, 3, 5, 10]:

            event_col = f"forward_return_{h}"
            baseline_col = f"baseline_return_{h}"

            event_returns = (
                pd.to_numeric(
                    regime_events[event_col],
                    errors="coerce",
                )
                .dropna()
            )

            baseline_returns = (
                pd.to_numeric(
                    regime_baseline[baseline_col],
                    errors="coerce",
                )
                .dropna()
            )

            if len(event_returns) == 0:
                continue

            event_mean = event_returns.mean()
            baseline_mean = baseline_returns.mean()

            trend_comparison.append({
                "period": period,
                "trend_regime": regime,
                "horizon": h,
                "event_n": len(event_returns),
                "baseline_n": len(baseline_returns),
                "event_mean": event_mean,
                "baseline_mean": baseline_mean,
                "mean_difference": (
                    event_mean - baseline_mean
                ),
                "event_median": event_returns.median(),
                "baseline_median": baseline_returns.median(),
                "event_win_rate": (
                    event_returns > 0
                ).mean(),
                "baseline_win_rate": (
                    baseline_returns > 0
                ).mean(),
            })

trend_comparison = pd.DataFrame(
    trend_comparison
)

trend_comparison

,period,trend_regime,horizon,event_n,baseline_n,event_mean,baseline_mean,mean_difference,event_median,baseline_median,event_win_rate,baseline_win_rate
0,development,downtrend,1,23,1104,-0.002395,-0.000443,-0.001952,-0.003289,0.000104,0.347826,0.503623
1,development,downtrend,3,23,1104,0.000042,0.000346,-0.000304,0.003988,0.002241,0.608696,0.535326
2,development,downtrend,5,23,1104,0.003548,0.000999,0.002549,0.005395,0.003048,0.521739,0.543478
3,development,downtrend,10,23,1104,-0.009706,0.001961,-0.011667,-0.004684,0.005925,0.478261,0.553442
4,development,uptrend,1,13,897,-0.000991,-0.000621,-0.000370,0.000874,-0.001047,0.615385,0.450390
5,development,uptrend,3,13,897,0.001513,-0.000515,0.002028,-0.001246,-0.000801,0.461538,0.482720
6,development,uptrend,5,13,897,0.016234,-0.000347,0.016581,0.016085,0.000222,0.692308,0.506132
7,development,uptrend,10,13,897,0.017986,0.000688,0.017298,0.024073,0.000423,0.538462,0.507246
8,oos,downtrend,1,13,1286,0.005912,-0.000605,0.006518,0.008643,-0.000425,0.769231,0.473561
9,oos,downtrend,3,13,1284,-0.003483,0.000141,-0.003624,0.005467,0.000962,0.538462,0.528816


**3. Important: check the period split**

In [141]:
print(
    baseline_trend.groupby(
        ["period", "trend_regime"]
    ).size()
)

period       trend_regime
development  downtrend       1104
             uptrend          897
oos          downtrend       1287
             uptrend         1247
dtype: int64


**Phase: Interaction Analysis**

In [142]:
print(df.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close']


In [143]:
valid_regime = (
    df_trend["trend_regime"].notna()
    & regime_df["volatility_regime"].notna()
)

df_trend.loc[valid_regime, "combined_regime"] = (
    df_trend.loc[valid_regime, "trend_regime"].astype(str)
    + "_"
    + regime_df.loc[valid_regime, "volatility_regime"].astype(str)
)

df_trend.loc[~valid_regime, "combined_regime"] = pd.NA

In [144]:
df_trend["combined_regime"].value_counts(dropna=False)

combined_regime
uptrend_low         1130
downtrend_high       995
downtrend_medium     693
uptrend_medium       589
downtrend_low        582
uptrend_high         344
NaN                  252
Name: count, dtype: int64

In [145]:
pd.crosstab(
    df_trend["trend_regime"],
    regime_df["volatility_regime"],
    dropna=False,
)

volatility_regime,high,low,medium,NaN
trend_regime,,,,
downtrend,995,582,693,121
uptrend,344,1130,589,81
NaN,0,0,0,50


**Then attach the regime to events**

In [146]:
events = events.drop(
    columns=["volatility_regime"],
    errors="ignore",
)

events = events.merge(
    regime_df[["Date", "volatility_regime"]],
    on="Date",
    how="left",
)

In [147]:
events.columns.tolist()

['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'event_return',
 'entry_date_1',
 'entry_price_1',
 'forward_return_1',
 'entry_date_3',
 'entry_price_3',
 'forward_return_3',
 'entry_date_5',
 'entry_price_5',
 'forward_return_5',
 'entry_date_10',
 'entry_price_10',
 'forward_return_10',
 'period',
 'trend_regime',
 'volatility_regime']

In [148]:
events["combined_regime"] = pd.NA

mask = (
    events["trend_regime"].notna()
    & events["volatility_regime"].notna()
)

events.loc[mask, "combined_regime"] = (
    events.loc[mask, "trend_regime"].astype(str)
    + "_"
    + events.loc[mask, "volatility_regime"].astype(str)
)

**Build the interaction table**

In [149]:
interaction_results = []

for (period, regime), group in events.groupby(
    ["period", "combined_regime"],
    dropna=True,
):

    row = {
        "period": period,
        "combined_regime": regime,
        "event_n": len(group),
    }

    for h in [1, 3, 5, 10]:
        col = f"forward_return_{h}"

        values = pd.to_numeric(
            group[col],
            errors="coerce",
        ).dropna()

        row[f"event_n_{h}"] = len(values)

        if len(values) > 0:
            row[f"event_mean_{h}"] = values.mean()
            row[f"event_median_{h}"] = values.median()
            row[f"event_win_rate_{h}"] = (
                (values > 0).mean()
            )
        else:
            row[f"event_mean_{h}"] = np.nan
            row[f"event_median_{h}"] = np.nan
            row[f"event_win_rate_{h}"] = np.nan

    interaction_results.append(row)

interaction_results = pd.DataFrame(
    interaction_results
)

In [150]:
interaction_results

,period,combined_regime,event_n,event_n_1,event_mean_1,event_median_1,event_win_rate_1,event_n_3,event_mean_3,event_median_3,event_win_rate_3,event_n_5,event_mean_5,event_median_5,event_win_rate_5,event_n_10,event_mean_10,event_median_10,event_win_rate_10
0,development,downtrend_high,11,11,-0.003650,-0.003289,0.272727,11,-0.001989,0.006782,0.818182,11,-0.000692,0.009613,0.545455,11,-0.024581,-0.007553,0.363636
1,development,downtrend_low,2,2,-0.023912,-0.023912,0.000000,2,-0.029161,-0.029161,0.000000,2,-0.039540,-0.039540,0.000000,2,-0.017620,-0.017620,0.500000
2,development,downtrend_medium,2,2,-0.010980,-0.010980,0.500000,2,0.033516,0.033516,0.500000,2,0.011978,0.011978,0.500000,2,0.008440,0.008440,0.500000
3,development,uptrend_high,1,1,0.027825,0.027825,1.000000,1,0.047417,0.047417,1.000000,1,0.012704,0.012704,1.000000,1,-0.043476,-0.043476,0.000000
4,development,uptrend_low,4,4,0.006194,0.008342,0.750000,4,0.011296,0.011058,0.750000,4,0.031729,0.035932,0.750000,4,0.036487,0.029667,0.500000
5,development,uptrend_medium,5,5,0.005406,0.004422,0.600000,5,0.005072,-0.001246,0.400000,5,0.008021,0.016085,0.600000,5,0.036881,0.065784,0.800000
6,oos,downtrend_high,12,12,0.005593,0.008331,0.750000,12,-0.005241,-0.001545,0.500000,12,-0.000057,0.011831,0.583333,12,-0.009073,0.015288,0.666667
7,oos,downtrend_medium,1,1,0.009739,0.009739,1.000000,1,0.017609,0.017609,1.000000,1,-0.004769,-0.004769,0.000000,1,0.034783,0.034783,1.000000
8,oos,uptrend_high,1,1,0.004016,0.004016,1.000000,1,0.036939,0.036939,1.000000,1,0.016024,0.016024,1.000000,1,0.015440,0.015440,1.000000
9,oos,uptrend_medium,1,1,0.006928,0.006928,1.000000,1,0.028085,0.028085,1.000000,1,0.041795,0.041795,1.000000,1,0.061752,0.061752,1.000000


**But we need a baseline for each regime**

In [151]:
baseline = baseline.merge(
    events[
        [
            "Date", "trend_regime",
            "volatility_regime",
            "combined_regime",
        ]
    ], on="Date", how="left",
)

In [152]:
baseline["period"] = np.where(
    baseline["Date"] < pd.Timestamp("2019-01-01"),
    "development", "oos",
)

**Sample-size rule**

In [153]:
MIN_EVENT_N = 10

In [154]:
result["sufficient_events"] = (
    result["event_n"] >= MIN_EVENT_N
)

**Phase: Trend × Volatility Interaction**

1. Create the combined regime

In [155]:
events["trend_volatility_regime"] = (
    events["trend_regime"].astype("string")
    + "_"
    + events["volatility_regime"].astype("string")
)

events["trend_volatility_regime"].value_counts(dropna=False)

trend_volatility_regime
downtrend_high      23
<NA>                13
uptrend_medium       6
uptrend_low          4
downtrend_medium     3
downtrend_low        2
uptrend_high         2
Name: count, dtype: Int64

2. Add the combined regime to events

In [156]:
events = events.copy()

regime_lookup = events[
    ["Date", "trend_regime", "volatility_regime"]
].copy()

events = events.merge(
    regime_lookup,
    on="Date",
    how="left",
    suffixes=("", "_df")
)

In [157]:
events["trend_volatility_regime"] = (
    events["trend_regime"].astype("string")
    + "_"
    + events["volatility_regime"].astype("string")
)

In [158]:
events[
    [
        "Date", "period",
        "trend_regime",
        "volatility_regime",
        "trend_volatility_regime",
    ]
].head()

,Date,period,trend_regime,volatility_regime,trend_volatility_regime
0,2007-10-18,development,NaN,NaN,<NA>
1,2007-11-21,development,NaN,NaN,<NA>
2,2007-12-17,development,uptrend,NaN,<NA>
3,2008-01-18,development,uptrend,NaN,<NA>
4,2008-02-07,development,downtrend,NaN,<NA>


3. Check sample sizes FIRST

In [159]:
interaction_counts = (
    events
    .dropna(subset=["trend_volatility_regime"])
    .groupby(
        ["period", "trend_volatility_regime"]
    )
    .size()
    .reset_index(name="event_n")
)

interaction_counts

,period,trend_volatility_regime,event_n
0,development,downtrend_high,11
1,development,downtrend_low,2
2,development,downtrend_medium,2
3,development,uptrend_high,1
4,development,uptrend_low,4
5,development,uptrend_medium,5
6,oos,downtrend_high,12
7,oos,downtrend_medium,1
8,oos,uptrend_high,1
9,oos,uptrend_medium,1


In [160]:
events[
    ["period", "trend_volatility_regime"]
].value_counts(dropna=False)

period       trend_volatility_regime
development  <NA>                       13
oos          downtrend_high             12
development  downtrend_high             11
             uptrend_medium              5
             uptrend_low                 4
             downtrend_medium            2
             downtrend_low               2
             uptrend_high                1
oos          uptrend_medium              1
             uptrend_high                1
             downtrend_medium            1
Name: count, dtype: int64

In [161]:
events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

In [162]:
events.groupby("period")["Date"].agg(["min", "max", "count"])

,min,max,count
period,,,
development,2007-10-18,2016-02-11,38
oos,2020-02-28,2026-03-19,15


In [163]:
split_date = events.loc[
    events["period"].eq("oos"), "Date"
].min()

split_date

Timestamp('2020-02-28 00:00:00')

In [164]:
df["period"] = np.where(
    df["Date"] < split_date,
    "development",
    "oos"
)

In [165]:
df["period"].value_counts()

period
development    3042
oos            1543
Name: count, dtype: int64

In [166]:
df.groupby("period")["Date"].agg(["min", "max", "count"])

,min,max,count
period,,,
development,2007-09-17,2020-02-27,3042
oos,2020-02-28,2026-05-29,1543


4. Run interaction analysis

Use the same forward-return columns:


In [167]:
horizons = [1, 3, 5, 10]
MIN_EVENTS = 10
interaction_results = []

for (period, regime), group in (
    events
    .dropna(subset=["trend_volatility_regime"])
    .groupby(["period", "trend_volatility_regime"])
):
    
    for h in horizons:
        col = f"forward_return_{h}"

        valid = group[col].dropna()

        event_n = len(valid)

        if event_n == 0:
            continue

        interaction_results.append({
            "period": period,
            "regime": regime,
            "horizon": h,
            "event_n": event_n,
            "event_mean": valid.mean(),
            "event_median": valid.median(),
            "event_win_rate": (valid > 0).mean(),
            "sufficient_events": event_n >= MIN_EVENTS,
        })

interaction_analysis = pd.DataFrame(
    interaction_results
)

interaction_analysis

,period,regime,horizon,event_n,event_mean,event_median,event_win_rate,sufficient_events
0,development,downtrend_high,1,11,-0.003650,-0.003289,0.272727,True
1,development,downtrend_high,3,11,-0.001989,0.006782,0.818182,True
2,development,downtrend_high,5,11,-0.000692,0.009613,0.545455,True
3,development,downtrend_high,10,11,-0.024581,-0.007553,0.363636,True
4,development,downtrend_low,1,2,-0.023912,-0.023912,0.000000,False
5,development,downtrend_low,3,2,-0.029161,-0.029161,0.000000,False
6,development,downtrend_low,5,2,-0.039540,-0.039540,0.000000,False
7,development,downtrend_low,10,2,-0.017620,-0.017620,0.500000,False
8,development,downtrend_medium,1,2,-0.010980,-0.010980,0.500000,False
9,development,downtrend_medium,3,2,0.033516,0.033516,0.500000,False


In [170]:
print("DF:")
print(df.columns.tolist())

print("\nBASELINE:")
print(baseline.columns.tolist())

print("\nEVENTS:")
print(events.columns.tolist())

print("\nEVENT PERIOD:")
print(events["period"].value_counts())

print("\nDF PERIOD:")
print(df["period"].value_counts())

print("\nEVENT REGIMES:")
print(
    events[
        ["period", "trend_regime", "volatility_regime"]
    ].value_counts(dropna=False)
)

DF:
['Date', 'Open', 'High', 'Low', 'Close', 'period']

BASELINE:
['Date', 'baseline_return_1', 'baseline_return_3', 'baseline_return_5', 'baseline_return_10', 'trend_regime_x', 'volatility_regime_x', 'combined_regime', 'period_x', 'period_y', 'trend_regime_y', 'volatility_regime_y']

EVENTS:
['Date', 'Open', 'High', 'Low', 'Close', 'event_return', 'entry_date_1', 'entry_price_1', 'forward_return_1', 'entry_date_3', 'entry_price_3', 'forward_return_3', 'entry_date_5', 'entry_price_5', 'forward_return_5', 'entry_date_10', 'entry_price_10', 'forward_return_10', 'period', 'trend_regime', 'volatility_regime', 'combined_regime', 'trend_volatility_regime', 'trend_regime_df', 'volatility_regime_df']

EVENT PERIOD:
period
development    38
oos            15
Name: count, dtype: int64

DF PERIOD:
period
development    3042
oos            1543
Name: count, dtype: int64

EVENT REGIMES:
period       trend_regime  volatility_regime
oos          downtrend     high                 12
development  down

1. Rebuild baseline cleanly

In [171]:
baseline = calculate_baseline_returns(
    df=df,
    holding_periods=(1, 3, 5, 10),
)

In [172]:
baseline = baseline[
    [
        "Date",
        "baseline_return_1",
        "baseline_return_3",
        "baseline_return_5",
        "baseline_return_10",
    ]
].copy()

**recreate volatility regime**

In [173]:
import numpy as np
import pandas as pd

df = df.sort_values("Date").reset_index(drop=True)

returns = df["Close"].pct_change()

df["rolling_volatility"] = (
    returns
    .rolling(window=20, min_periods=20)
    .std()
)

Create the three regimes using the historical distribution:

In [174]:
q33 = df["rolling_volatility"].quantile(1/3)
q67 = df["rolling_volatility"].quantile(2/3)

df["volatility_regime"] = pd.cut(
    df["rolling_volatility"],
    bins=[-np.inf, q33, q67, np.inf],
    labels=["low", "medium", "high"],
)

In [175]:
df["volatility_regime"].value_counts(dropna=False)

volatility_regime
low       1522
high      1522
medium    1521
NaN         20
Name: count, dtype: int64

**recreate trend regime**

In [176]:
import numpy as np
import pandas as pd

df = df.copy()

# Ensure date ordering
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# --------------------------------------------------
# 1. Trend regime
# --------------------------------------------------

lookback = 20

df["trend_ma"] = (
    df["Close"]
    .rolling(lookback, min_periods=lookback)
    .mean()
)

df["trend_regime"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)

valid = df["trend_ma"].notna()

df.loc[
    valid & (df["Close"] >= df["trend_ma"]),
    "trend_regime"
] = "uptrend"

df.loc[
    valid & (df["Close"] < df["trend_ma"]),
    "trend_regime"
] = "downtrend"


# --------------------------------------------------
# 2. Volatility regime
# --------------------------------------------------

returns = df["Close"].pct_change()

vol_window = 20

df["rolling_volatility"] = (
    returns
    .rolling(vol_window, min_periods=vol_window)
    .std()
)

# Use quantiles only on valid observations
vol_valid = df["rolling_volatility"].dropna()

low_cutoff = vol_valid.quantile(1 / 3)
high_cutoff = vol_valid.quantile(2 / 3)

df["volatility_regime"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)

valid_vol = df["rolling_volatility"].notna()

df.loc[
    valid_vol &
    (df["rolling_volatility"] <= low_cutoff),
    "volatility_regime"
] = "low"

df.loc[
    valid_vol &
    (df["rolling_volatility"] > low_cutoff) &
    (df["rolling_volatility"] <= high_cutoff),
    "volatility_regime"
] = "medium"

df.loc[
    valid_vol &
    (df["rolling_volatility"] > high_cutoff),
    "volatility_regime"
] = "high"


# --------------------------------------------------
# 3. Combined regime
# --------------------------------------------------

df["trend_volatility_regime"] = (
    df["trend_regime"]
    + "_"
    + df["volatility_regime"]
)

df.loc[
    df["trend_regime"].isna() |
    df["volatility_regime"].isna(),
    "trend_volatility_regime"
] = pd.NA

In [177]:
print(df.columns.tolist())

print("\nTrend:")
print(df["trend_regime"].value_counts(dropna=False))

print("\nVolatility:")
print(df["volatility_regime"].value_counts(dropna=False))

print("\nCombined:")
print(df["trend_volatility_regime"].value_counts(dropna=False))

['Date', 'Open', 'High', 'Low', 'Close', 'period', 'rolling_volatility', 'volatility_regime', 'trend_ma', 'trend_regime', 'trend_volatility_regime']

Trend:
trend_regime
uptrend      2721
downtrend    1845
<NA>           19
Name: count, dtype: Int64

Volatility:
volatility_regime
high      1522
low       1522
medium    1521
<NA>        20
Name: count, dtype: Int64

Combined:
trend_volatility_regime
uptrend_low         1075
uptrend_medium       845
uptrend_high         800
downtrend_high       722
downtrend_medium     676
downtrend_low        447
<NA>                  20
Name: count, dtype: Int64


Recreate a clean baseline:

In [178]:
BASELINE = baseline.copy()

BASELINE = BASELINE[
    [
        "Date",
        "baseline_return_1",
        "baseline_return_3",
        "baseline_return_5",
        "baseline_return_10",
    ]
].copy()

BASELINE["Date"] = pd.to_datetime(BASELINE["Date"])

Then attach regimes once:

In [179]:
regime_columns = df[
    [
        "Date", "period",
        "trend_regime",
        "volatility_regime",
        "trend_volatility_regime",
    ]
].copy()

BASELINE = BASELINE.merge(
    regime_columns,
    on="Date",
    how="left",
    validate="one_to_one",
)

In [180]:
print(df.columns.tolist())
print(BASELINE.columns.tolist())
print(events.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'period', 'rolling_volatility', 'volatility_regime', 'trend_ma', 'trend_regime', 'trend_volatility_regime']
['Date', 'baseline_return_1', 'baseline_return_3', 'baseline_return_5', 'baseline_return_10', 'period', 'trend_regime', 'volatility_regime', 'trend_volatility_regime']
['Date', 'Open', 'High', 'Low', 'Close', 'event_return', 'entry_date_1', 'entry_price_1', 'forward_return_1', 'entry_date_3', 'entry_price_3', 'forward_return_3', 'entry_date_5', 'entry_price_5', 'forward_return_5', 'entry_date_10', 'entry_price_10', 'forward_return_10', 'period', 'trend_regime', 'volatility_regime', 'combined_regime', 'trend_volatility_regime', 'trend_regime_df', 'volatility_regime_df']


In [181]:
print(BASELINE["period"].value_counts(dropna=False))
print(BASELINE["trend_regime"].value_counts(dropna=False))
print(BASELINE["volatility_regime"].value_counts(dropna=False))

period
development    3042
oos            1543
Name: count, dtype: int64
trend_regime
uptrend      2721
downtrend    1845
<NA>           19
Name: count, dtype: Int64
volatility_regime
high      1522
low       1522
medium    1521
<NA>        20
Name: count, dtype: Int64


**Next phase: Clean regime-conditioned event study**

Step 1 — Remove merge artifacts

In [182]:
EVENTS = events.copy()

EVENTS = EVENTS.drop(
    columns=[
        "combined_regime",
        "trend_regime_df",
        "volatility_regime_df",
    ],
    errors="ignore",
)

In [183]:
print(EVENTS.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'event_return', 'entry_date_1', 'entry_price_1', 'forward_return_1', 'entry_date_3', 'entry_price_3', 'forward_return_3', 'entry_date_5', 'entry_price_5', 'forward_return_5', 'entry_date_10', 'entry_price_10', 'forward_return_10', 'period', 'trend_regime', 'volatility_regime', 'trend_volatility_regime']


Step 2 — Validate the event dataset

In [184]:
required_event_columns = [
    "Date",
    "event_return",
    "forward_return_1",
    "forward_return_3",
    "forward_return_5",
    "forward_return_10",
    "period",
    "trend_regime",
    "volatility_regime",
    "trend_volatility_regime",
]

missing = [
    col for col in required_event_columns
    if col not in EVENTS.columns
]

print("Missing columns:", missing)

Missing columns: []


In [185]:
print(EVENTS.shape)

print(
    EVENTS[
        [
            "period", "trend_regime",
            "volatility_regime",
            "trend_volatility_regime",
        ]
    ].value_counts(dropna=False)
)

(53, 22)
period       trend_regime  volatility_regime  trend_volatility_regime
oos          downtrend     high               downtrend_high             12
development  downtrend     high               downtrend_high             11
                           NaN                <NA>                        8
             uptrend       medium             uptrend_medium              5
                           low                uptrend_low                 4
                           NaN                <NA>                        3
             NaN           NaN                <NA>                        2
             downtrend     medium             downtrend_medium            2
                           low                downtrend_low               2
             uptrend       high               uptrend_high                1
oos          uptrend       medium             uptrend_medium              1
                           high               uptrend_high                1
         

Step 3 — Check event/regime consistency

In [186]:
regime_consistency = EVENTS[
    [
        "Date", "period",
        "trend_regime",
        "volatility_regime",
        "trend_volatility_regime",
    ]
].copy()

print(regime_consistency.head(10))

        Date       period trend_regime volatility_regime  \
0 2007-10-18  development          NaN               NaN   
1 2007-11-21  development          NaN               NaN   
2 2007-12-17  development      uptrend               NaN   
3 2008-01-18  development      uptrend               NaN   
4 2008-02-07  development    downtrend               NaN   
5 2008-03-03  development    downtrend               NaN   
6 2008-03-13  development    downtrend               NaN   
7 2008-03-31  development    downtrend               NaN   
8 2008-06-20  development    downtrend               NaN   
9 2008-07-01  development    downtrend               NaN   

  trend_volatility_regime  
0                    <NA>  
1                    <NA>  
2                    <NA>  
3                    <NA>  
4                    <NA>  
5                    <NA>  
6                    <NA>  
7                    <NA>  
8                    <NA>  
9                    <NA>  


Now independently compare the event regime against the canonical df.

In [187]:
canonical_regimes = df[
    [
        "Date", "period",
        "trend_regime",
        "volatility_regime",
        "trend_volatility_regime",
    ]
].copy()

check = EVENTS[
    [
        "Date", "period",
        "trend_regime",
        "volatility_regime",
        "trend_volatility_regime",
    ]
].merge(
    canonical_regimes,
    on="Date", how="left",
    suffixes=("_event", "_df"),
    validate="one_to_one",
)

Check discrepancies:

In [188]:
for col in [
    "period", "trend_regime",
    "volatility_regime",
    "trend_volatility_regime",
]:
    mismatch = (
        check[f"{col}_event"].astype("string")
        != check[f"{col}_df"].astype("string")
    )

    print(
        col,
        "mismatches:",
        mismatch.sum()
    )

period mismatches: 0
trend_regime mismatches: 16
volatility_regime mismatches: 14
trend_volatility_regime mismatches: 17


**Step 4 — Validate regime counts**

In [189]:
print("EVENT REGIMES")
print(
    EVENTS["trend_regime"]
    .value_counts(dropna=False)
)

print(
    EVENTS["volatility_regime"]
    .value_counts(dropna=False)
)

print(
    EVENTS["trend_volatility_regime"]
    .value_counts(dropna=False)
)

EVENT REGIMES
trend_regime
downtrend    36
uptrend      15
NaN           2
Name: count, dtype: int64
volatility_regime
high      25
NaN       13
medium     9
low        6
Name: count, dtype: int64
trend_volatility_regime
downtrend_high      23
<NA>                13
uptrend_medium       6
uptrend_low          4
downtrend_medium     3
downtrend_low        2
uptrend_high         2
Name: count, dtype: Int64


In [190]:
print("\nEVENT PERIOD × REGIME")

print(
    EVENTS.groupby(
        [
            "period",
            "trend_regime",
            "volatility_regime",
        ],
        dropna=False
    ).size()
)


EVENT PERIOD × REGIME
period       trend_regime  volatility_regime
development  downtrend     high                 11
                           low                   2
                           medium                2
                           NaN                   8
             uptrend       high                  1
                           low                   4
                           medium                5
                           NaN                   3
             NaN           NaN                   2
oos          downtrend     high                 12
                           medium                1
             uptrend       high                  1
                           medium                1
dtype: int64


**Step 5 — Recalculate regime-conditioned returns**

In [191]:
horizons = [1, 3, 5, 10]

regime_results = []

for (
    period,
    trend_regime,
    volatility_regime,
), group in EVENTS.groupby(
    [
        "period",
        "trend_regime",
        "volatility_regime",
    ],
    dropna=False,
):

    for horizon in horizons:

        col = f"forward_return_{horizon}"

        values = pd.to_numeric(
            group[col],
            errors="coerce"
        ).dropna()

        if len(values) == 0:
            continue

        regime_results.append({
            "period": period,
            "trend_regime": trend_regime,
            "volatility_regime": volatility_regime,
            "horizon": horizon,
            "event_n": len(values),
            "event_mean": values.mean(),
            "event_median": values.median(),
            "event_win_rate": (values > 0).mean(),
        })

regime_event_results = pd.DataFrame(
    regime_results
)

regime_event_results

,period,trend_regime,volatility_regime,horizon,event_n,event_mean,event_median,event_win_rate
0,development,downtrend,high,1,11,-0.003650,-0.003289,0.272727
1,development,downtrend,high,3,11,-0.001989,0.006782,0.818182
2,development,downtrend,high,5,11,-0.000692,0.009613,0.545455
3,development,downtrend,high,10,11,-0.024581,-0.007553,0.363636
4,development,downtrend,low,1,2,-0.023912,-0.023912,0.000000
5,development,downtrend,low,3,2,-0.029161,-0.029161,0.000000
6,development,downtrend,low,5,2,-0.039540,-0.039540,0.000000
7,development,downtrend,low,10,2,-0.017620,-0.017620,0.500000
8,development,downtrend,medium,1,2,-0.010980,-0.010980,0.500000
9,development,downtrend,medium,3,2,0.033516,0.033516,0.500000


**Step 6 — Attach the correct baseline**

In [192]:
baseline_results = []

for (
    period,
    trend_regime,
    volatility_regime,
), group in BASELINE.groupby(
    [
        "period",
        "trend_regime",
        "volatility_regime",
    ],
    dropna=False,
):

    for horizon in horizons:

        col = f"baseline_return_{horizon}"

        values = pd.to_numeric(
            group[col],
            errors="coerce"
        ).dropna()

        if len(values) == 0:
            continue

        baseline_results.append({
            "period": period,
            "trend_regime": trend_regime,
            "volatility_regime": volatility_regime,
            "horizon": horizon,
            "baseline_n": len(values),
            "baseline_mean": values.mean(),
            "baseline_median": values.median(),
            "baseline_win_rate": (values > 0).mean(),
        })

regime_baseline_results = pd.DataFrame(
    baseline_results
)

**Step 7 — Calculate abnormal/event difference**

In [193]:
regime_analysis = regime_event_results.merge(
    regime_baseline_results,
    on=[
        "period", "trend_regime",
        "volatility_regime", "horizon",
    ],
    how="left",
)

In [194]:
regime_analysis["mean_difference"] = (
    regime_analysis["event_mean"]
    - regime_analysis["baseline_mean"]
)

In [195]:
regime_analysis["win_rate_difference"] = (
    regime_analysis["event_win_rate"]
    - regime_analysis["baseline_win_rate"]
)

In [196]:
regime_analysis[
    [
        "period",
        "trend_regime",
        "volatility_regime",
        "horizon",
        "event_n",
        "baseline_n",
        "event_mean",
        "baseline_mean",
        "mean_difference",
        "event_win_rate",
        "baseline_win_rate",
        "win_rate_difference",
    ]
]

,period,trend_regime,volatility_regime,horizon,event_n,baseline_n,event_mean,baseline_mean,mean_difference,event_win_rate,baseline_win_rate,win_rate_difference
0,development,downtrend,high,1,11,546.0,-0.003650,-0.001091,-0.002560,0.272727,0.481685,-0.208958
1,development,downtrend,high,3,11,546.0,-0.001989,-0.000763,-0.001227,0.818182,0.501832,0.316350
2,development,downtrend,high,5,11,546.0,-0.000692,0.000328,-0.001020,0.545455,0.531136,0.014319
3,development,downtrend,high,10,11,546.0,-0.024581,0.002446,-0.027027,0.363636,0.545788,-0.182151
4,development,downtrend,low,1,2,228.0,-0.023912,-0.000227,-0.023685,0.000000,0.442982,-0.442982
5,development,downtrend,low,3,2,228.0,-0.029161,0.002994,-0.032155,0.000000,0.574561,-0.574561
6,development,downtrend,low,5,2,228.0,-0.039540,0.006439,-0.045979,0.000000,0.671053,-0.671053
7,development,downtrend,low,10,2,228.0,-0.017620,0.011532,-0.029152,0.500000,0.679825,-0.179825
8,development,downtrend,medium,1,2,481.0,-0.010980,-0.000852,-0.010127,0.500000,0.459459,0.040541
9,development,downtrend,medium,3,2,481.0,0.033516,-0.000659,0.034176,0.500000,0.496881,0.003119


**Step 8 — Critical sample-size filter**

In [197]:
regime_analysis["sufficient_events"] = (
    regime_analysis["event_n"] >= 10
)

In [198]:
regime_analysis[
    regime_analysis["sufficient_events"]
]

,period,trend_regime,volatility_regime,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference,win_rate_difference,sufficient_events
0,development,downtrend,high,1,11,-0.003650,-0.003289,0.272727,546.0,-0.001091,-0.000878,0.481685,-0.002560,-0.208958,True
1,development,downtrend,high,3,11,-0.001989,0.006782,0.818182,546.0,-0.000763,0.000201,0.501832,-0.001227,0.316350,True
2,development,downtrend,high,5,11,-0.000692,0.009613,0.545455,546.0,0.000328,0.003040,0.531136,-0.001020,0.014319,True
3,development,downtrend,high,10,11,-0.024581,-0.007553,0.363636,546.0,0.002446,0.005263,0.545788,-0.027027,-0.182151,True
36,oos,downtrend,high,1,12,0.005593,0.008331,0.750000,176.0,-0.000007,0.000888,0.545455,0.005600,0.204545,True
37,oos,downtrend,high,3,12,-0.005241,-0.001545,0.500000,176.0,0.000117,0.004222,0.551136,-0.005357,-0.051136,True
38,oos,downtrend,high,5,12,-0.000057,0.011831,0.583333,176.0,0.000739,0.005729,0.562500,-0.000796,0.020833,True
39,oos,downtrend,high,10,12,-0.009073,0.015288,0.666667,176.0,0.008539,0.016692,0.659091,-0.017612,0.007576,True


In [199]:
regime_analysis[
    ~regime_analysis["sufficient_events"]
]

,period,trend_regime,volatility_regime,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference,win_rate_difference,sufficient_events
4,development,downtrend,low,1,2,-0.023912,-0.023912,0.000000,228.0,-0.000227,-0.000643,0.442982,-0.023685,-0.442982,False
5,development,downtrend,low,3,2,-0.029161,-0.029161,0.000000,228.0,0.002994,0.003253,0.574561,-0.032155,-0.574561,False
6,development,downtrend,low,5,2,-0.039540,-0.039540,0.000000,228.0,0.006439,0.008209,0.671053,-0.045979,-0.671053,False
7,development,downtrend,low,10,2,-0.017620,-0.017620,0.500000,228.0,0.011532,0.012055,0.679825,-0.029152,-0.179825,False
8,development,downtrend,medium,1,2,-0.010980,-0.010980,0.500000,481.0,-0.000852,-0.001353,0.459459,-0.010127,0.040541,False
9,development,downtrend,medium,3,2,0.033516,0.033516,0.500000,481.0,-0.000659,-0.000285,0.496881,0.034176,0.003119,False
10,development,downtrend,medium,5,2,0.011978,0.011978,0.500000,481.0,-0.000713,-0.000366,0.496881,0.012691,0.003119,False
11,development,downtrend,medium,10,2,0.008440,0.008440,0.500000,481.0,-0.001819,-0.000036,0.498960,0.010259,0.001040,False
12,development,downtrend,NaN,1,8,0.006856,-0.000733,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,False
13,development,downtrend,NaN,3,8,0.001767,-0.006023,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,False
